# 202 — TCGA DNA Methylation Quality Control

## Objective

Perform analytical quality control of the TCGA SeSAMe DNA-methylation beta-value files included in the frozen HM27/HM450 primary-tumor cohort.

This notebook evaluates methylation-file structure, sample-level completeness, probe-level missingness, beta-value validity, and platform-specific technical variation before multi-omic integration.

## Inputs

- Frozen TCGA HM27/HM450 methylation file index.
- Frozen TCGA methylation cohort definition.
- Locally validated SeSAMe beta-value files.
- RNA-seq-QC-annotated candidate-pair inventory produced by notebook 201.

## Main tasks

1. Reconstruct the unique methylation-file inventory represented in the RNA-seq-QC-eligible candidate pairs.
2. Confirm file availability and analytical structure.
3. Characterize sample-level missingness and beta-value distributions.
4. Evaluate probe identity, completeness, and platform-specific coverage.
5. Define transparent sample- and probe-level methylation-QC criteria.
6. Propagate methylation-QC status to the candidate-pair inventory.
7. Produce versioned artifacts for downstream multi-omic integration.

## Scope boundaries

This notebook performs technical and analytical methylation quality control only.

It does not:

- integrate methylation with RNA-seq;
- aggregate probes to genes or promoters;
- perform batch correction;
- normalize jointly across HM27 and HM450;
- select a final unique multi-omic sample per case;
- infer biological programs or methylation-expression associations.

HM27 and HM450 are evaluated separately whenever platform-specific structure may affect the results. Any shared probe space required for integration will be defined explicitly and will not be created through naïve platform pooling.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Inputs and project paths
# =============================================================================

# -----------------------------------------------------------------------------
# Canonical project root
# -----------------------------------------------------------------------------

PROJECT_ROOT = Paths.root


# -----------------------------------------------------------------------------
# Authoritative candidate-pair inventory from notebook 201
# -----------------------------------------------------------------------------

RNA_QC_ANNOTATED_PAIR_INVENTORY_PATH = (
    Paths.metadata
    / (
        "tcga_primary_tumor_rnaseq_methylation_"
        "rna_qc_annotated_candidate_pair_inventory.csv"
    )
)


# -----------------------------------------------------------------------------
# Frozen TCGA methylation cohort inputs
# -----------------------------------------------------------------------------

METHYLATION_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_methylation"
)

METHYLATION_FILE_INDEX_PATH = (
    METHYLATION_MANIFEST_DIR
    / (
        "gdc_file_index_tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450.tsv"
    )
)

METHYLATION_COHORT_FREEZE_PATH = (
    METHYLATION_MANIFEST_DIR
    / (
        "gdc_cohort_freeze_tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450.json"
    )
)


# -----------------------------------------------------------------------------
# Local methylation payload directory
# -----------------------------------------------------------------------------

METHYLATION_DOWNLOAD_DIR = (
    Paths.tcga
    / "methylation"
)


# -----------------------------------------------------------------------------
# Downstream output locations
# -----------------------------------------------------------------------------

METHYLATION_QC_OUTPUT_DIR = Paths.qc
METHYLATION_DATA_OUTPUT_DIR = Paths.methylation
METHYLATION_METADATA_OUTPUT_DIR = Paths.metadata


# -----------------------------------------------------------------------------
# Path summary
# -----------------------------------------------------------------------------

print("TCGA DNA-methylation quality-control paths resolved.")
print(
    f"Project root:                 "
    f"{project_relative_path(PROJECT_ROOT)}"
)
print(
    f"RNA-QC pair inventory:        "
    f"{project_relative_path(RNA_QC_ANNOTATED_PAIR_INVENTORY_PATH)}"
)
print(
    f"Methylation file index:       "
    f"{project_relative_path(METHYLATION_FILE_INDEX_PATH)}"
)
print(
    f"Methylation cohort freeze:    "
    f"{project_relative_path(METHYLATION_COHORT_FREEZE_PATH)}"
)
print(
    f"Methylation download dir:     "
    f"{project_relative_path(METHYLATION_DOWNLOAD_DIR)}"
)
print(
    f"QC output directory:          "
    f"{project_relative_path(METHYLATION_QC_OUTPUT_DIR)}"
)
print(
    f"Methylation output directory: "
    f"{project_relative_path(METHYLATION_DATA_OUTPUT_DIR)}"
)
print(
    f"Metadata output directory:    "
    f"{project_relative_path(METHYLATION_METADATA_OUTPUT_DIR)}"
)

TCGA DNA-methylation quality-control paths resolved.
Project root:                 .
RNA-QC pair inventory:        data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_rna_qc_annotated_candidate_pair_inventory.csv
Methylation file index:       config/manifests/tcga_methylation/gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv
Methylation cohort freeze:    config/manifests/tcga_methylation/gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json
Methylation download dir:     data/raw/tcga/methylation
QC output directory:          data/interim/qc
Methylation output directory: data/interim/methylation
Metadata output directory:    data/interim/metadata


In [3]:
# =============================================================================
# Validate required input locations
# =============================================================================

required_input_checks = {
    "rna_qc_pair_inventory_is_file": (
        RNA_QC_ANNOTATED_PAIR_INVENTORY_PATH.is_file()
    ),
    "methylation_file_index_is_file": (
        METHYLATION_FILE_INDEX_PATH.is_file()
    ),
    "methylation_cohort_freeze_is_file": (
        METHYLATION_COHORT_FREEZE_PATH.is_file()
    ),
    "methylation_download_directory_is_dir": (
        METHYLATION_DOWNLOAD_DIR.is_dir()
    ),
}

print("Required-input checks:")
for check_name, check_passed in required_input_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(required_input_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in required_input_checks.items()
        if not check_passed
    ]

    raise FileNotFoundError(
        "Required TCGA methylation-QC inputs are unavailable: "
        + ", ".join(failed_checks)
    )

Required-input checks:
rna_qc_pair_inventory_is_file: True
methylation_file_index_is_file: True
methylation_cohort_freeze_is_file: True
methylation_download_directory_is_dir: True


In [4]:
# =============================================================================
# Load authoritative methylation-QC inputs
# =============================================================================

rna_qc_pair_inventory = pd.read_csv(
    RNA_QC_ANNOTATED_PAIR_INVENTORY_PATH,
    low_memory=False,
)

methylation_file_index = pd.read_csv(
    METHYLATION_FILE_INDEX_PATH,
    sep="\t",
    dtype="string",
)

with METHYLATION_COHORT_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    methylation_cohort_freeze = json.load(handle)


print("Authoritative methylation-QC inputs loaded.")
print(
    f"RNA-QC pair inventory:         "
    f"{rna_qc_pair_inventory.shape}"
)
print(
    f"Methylation file index:        "
    f"{methylation_file_index.shape}"
)
print(
    f"Cohort-freeze top-level keys:  "
    f"{len(methylation_cohort_freeze):,}"
)

Authoritative methylation-QC inputs loaded.
RNA-QC pair inventory:         (10162, 51)
Methylation file index:        (10897, 25)
Cohort-freeze top-level keys:  12


In [5]:
# =============================================================================
# Inspect authoritative input schemas
# =============================================================================

schema_checks = {
    "pair_inventory_columns_are_unique": (
        rna_qc_pair_inventory.columns.is_unique
    ),
    "methylation_file_index_columns_are_unique": (
        methylation_file_index.columns.is_unique
    ),
    "cohort_freeze_is_dictionary": (
        isinstance(methylation_cohort_freeze, dict)
    ),
}

print("Input-schema checks:")
for check_name, check_passed in schema_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(schema_checks.values()):
    raise ValueError(
        "One or more authoritative input schemas are invalid."
    )


pair_column_tokens = (
    "project",
    "case",
    "sample",
    "portion",
    "methylation",
    "rna_qc",
    "pair_",
)

relevant_pair_columns = [
    column
    for column in rna_qc_pair_inventory.columns
    if any(
        token in column.lower()
        for token in pair_column_tokens
    )
]

print("\nRelevant candidate-pair columns:")
for column in relevant_pair_columns:
    print(f"- {column}")

print("\nMethylation file-index columns:")
for column in methylation_file_index.columns:
    print(f"- {column}")

print("\nCohort-freeze top-level keys:")
for key in methylation_cohort_freeze:
    print(f"- {key}")

Input-schema checks:
pair_inventory_columns_are_unique: True
methylation_file_index_columns_are_unique: True
cohort_freeze_is_dictionary: True

Relevant candidate-pair columns:
- project_id
- case_submitter_id
- sample_submitter_id
- rna_case_uuid
- methylation_case_uuid
- methylation_aliquot_uuid
- methylation_aliquot_submitter_id
- methylation_file_id
- methylation_file_name
- methylation_platform
- rna_project
- rna_sample_type_code
- rna_portion_number
- rna_sample_submitter_id_from_aliquot
- methylation_project
- methylation_tss
- methylation_participant
- methylation_sample_type_code
- methylation_vial
- methylation_portion_number
- methylation_analyte_code
- methylation_plate
- methylation_center_code
- methylation_sample_submitter_id_from_aliquot
- same_biospecimen_portion
- primary_pair_policy_status
- project_multidomain_review_candidate
- same_portion_analyte_comparator_count
- rna_qc_eligible_for_downstream_selection
- rna_qc_eligibility_status
- rna_qc_ineligibility_reason

In [6]:
# =============================================================================
# Validate fields required for the methylation candidate inventory
# =============================================================================

required_pair_columns = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "methylation_case_uuid",
    "methylation_aliquot_uuid",
    "methylation_aliquot_submitter_id",
    "methylation_file_id",
    "methylation_file_name",
    "methylation_platform",
    "pair_eligible_after_rna_qc",
    "pair_rna_qc_status",
]

missing_pair_columns = [
    column
    for column in required_pair_columns
    if column not in rna_qc_pair_inventory.columns
]

if missing_pair_columns:
    raise KeyError(
        "Required candidate-pair columns are missing: "
        + ", ".join(missing_pair_columns)
    )

pair_gate_checks = {
    "required_columns_are_present": not missing_pair_columns,
    "pair_gate_has_no_missing_values": (
        rna_qc_pair_inventory[
            "pair_eligible_after_rna_qc"
        ].notna().all()
    ),
    "pair_gate_is_boolean": (
        pd.api.types.is_bool_dtype(
            rna_qc_pair_inventory[
                "pair_eligible_after_rna_qc"
            ]
        )
    ),
}

print("Candidate-pair gate checks:")
for check_name, check_passed in pair_gate_checks.items():
    print(f"{check_name}: {check_passed}")

print("\nPair eligibility after RNA-seq QC:")
print(
    rna_qc_pair_inventory[
        "pair_eligible_after_rna_qc"
    ].value_counts(dropna=False)
)

print("\nPair RNA-seq-QC status:")
print(
    rna_qc_pair_inventory[
        "pair_rna_qc_status"
    ].value_counts(dropna=False)
)

if not all(pair_gate_checks.values()):
    raise ValueError(
        "The RNA-seq-QC candidate-pair gate is invalid."
    )

Candidate-pair gate checks:
required_columns_are_present: True
pair_gate_has_no_missing_values: True
pair_gate_is_boolean: True

Pair eligibility after RNA-seq QC:
pair_eligible_after_rna_qc
True     10158
False        4
Name: count, dtype: int64

Pair RNA-seq-QC status:
pair_rna_qc_status
Eligible candidate pair after RNA-seq QC    10158
Ineligible candidate pair — RNA-seq QC          4
Name: count, dtype: int64


In [7]:
# =============================================================================
# Construct the unique methylation candidate-file inventory
# =============================================================================

eligible_pair_inventory = (
    rna_qc_pair_inventory.loc[
        rna_qc_pair_inventory[
            "pair_eligible_after_rna_qc"
        ]
    ]
    .copy()
)

methylation_identity_columns = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "methylation_case_uuid",
    "methylation_aliquot_uuid",
    "methylation_aliquot_submitter_id",
    "methylation_file_id",
    "methylation_file_name",
    "methylation_platform",
]

identity_consistency = (
    eligible_pair_inventory
    .groupby("methylation_file_id")[
        methylation_identity_columns
    ]
    .nunique(dropna=False)
)

if identity_consistency.gt(1).any().any():
    inconsistent_columns = (
        identity_consistency
        .gt(1)
        .any()
    )

    raise ValueError(
        "Inconsistent methylation-file identity fields: "
        + ", ".join(
            inconsistent_columns[
                inconsistent_columns
            ].index
        )
    )

methylation_pair_counts = (
    eligible_pair_inventory[
        "methylation_file_id"
    ]
    .value_counts()
    .rename("eligible_candidate_pair_count")
)

methylation_candidate_inventory = (
    eligible_pair_inventory[
        methylation_identity_columns
    ]
    .drop_duplicates(
        subset="methylation_file_id",
        keep="first",
    )
    .merge(
        methylation_pair_counts,
        left_on="methylation_file_id",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "methylation_file_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Unique methylation candidate-file inventory constructed.")
print(f"Eligible candidate pairs:  {len(eligible_pair_inventory):,}")
print(f"Unique methylation files:  {len(methylation_candidate_inventory):,}")
print(
    f"Cases represented:         "
    f"{methylation_candidate_inventory['methylation_case_uuid'].nunique():,}"
)
print("\nFiles by platform:")
print(
    methylation_candidate_inventory[
        "methylation_platform"
    ].value_counts()
)
print("\nEligible candidate-pair multiplicity per methylation file:")
print(
    methylation_candidate_inventory[
        "eligible_candidate_pair_count"
    ].value_counts()
    .sort_index()
)

Unique methylation candidate-file inventory constructed.
Eligible candidate pairs:  10,158
Unique methylation files:  10,038
Cases represented:         9,973

Files by platform:
methylation_platform
Illumina Human Methylation 450    8417
Illumina Human Methylation 27     1621
Name: count, dtype: int64

Eligible candidate-pair multiplicity per methylation file:
eligible_candidate_pair_count
1    9918
2     120
Name: count, dtype: int64


In [8]:
# =============================================================================
# Match candidate methylation files to the frozen file index
# =============================================================================

frozen_index_subset = (
    methylation_file_index[
        [
            "file_id",
            "file_name",
            "md5",
            "file_size_bytes",
            "platform",
            "project_id",
            "case_uuid",
            "case_submitter_id",
            "sample_submitter_id",
            "aliquot_uuid",
            "aliquot_submitter_id",
        ]
    ]
    .rename(
        columns={
            "file_id": "methylation_file_id",
            "file_name": "index_file_name",
            "md5": "methylation_md5",
            "file_size_bytes": "methylation_file_size_bytes",
            "platform": "index_platform",
            "project_id": "index_project_id",
            "case_uuid": "index_case_uuid",
            "case_submitter_id": "index_case_submitter_id",
            "sample_submitter_id": "index_sample_submitter_id",
            "aliquot_uuid": "index_aliquot_uuid",
            "aliquot_submitter_id": "index_aliquot_submitter_id",
        }
    )
)

methylation_candidate_inventory = (
    methylation_candidate_inventory
    .merge(
        frozen_index_subset,
        on="methylation_file_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

identity_checks = {
    "all_candidate_files_match_frozen_index": (
        methylation_candidate_inventory["_merge"]
        .eq("both")
        .all()
    ),
    "file_names_match": (
        methylation_candidate_inventory["methylation_file_name"]
        .eq(methylation_candidate_inventory["index_file_name"])
        .all()
    ),
    "platforms_match": (
        methylation_candidate_inventory["methylation_platform"]
        .eq(methylation_candidate_inventory["index_platform"])
        .all()
    ),
    "projects_match": (
        methylation_candidate_inventory["project_id"]
        .eq(methylation_candidate_inventory["index_project_id"])
        .all()
    ),
    "case_uuids_match": (
        methylation_candidate_inventory["methylation_case_uuid"]
        .eq(methylation_candidate_inventory["index_case_uuid"])
        .all()
    ),
    "aliquot_uuids_match": (
        methylation_candidate_inventory["methylation_aliquot_uuid"]
        .eq(methylation_candidate_inventory["index_aliquot_uuid"])
        .all()
    ),
}

print("Frozen-index matching checks:")
for check_name, check_passed in identity_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(identity_checks.values()):
    raise ValueError(
        "Candidate methylation-file identity does not match "
        "the frozen file index."
    )

Frozen-index matching checks:
all_candidate_files_match_frozen_index: True
file_names_match: True
platforms_match: True
projects_match: True
case_uuids_match: True
aliquot_uuids_match: True


In [9]:
# =============================================================================
# Resolve and validate local methylation payload paths
# =============================================================================

methylation_candidate_inventory = (
    methylation_candidate_inventory
    .drop(columns="_merge")
)

methylation_candidate_inventory["methylation_local_path"] = [
    (
        METHYLATION_DOWNLOAD_DIR
        / file_id
        / file_name
    )
    for file_id, file_name in zip(
        methylation_candidate_inventory["methylation_file_id"],
        methylation_candidate_inventory["methylation_file_name"],
    )
]

local_path_checks = {
    "local_paths_are_unique": (
        methylation_candidate_inventory[
            "methylation_local_path"
        ].is_unique
    ),
    "all_candidate_payloads_are_available": (
        methylation_candidate_inventory[
            "methylation_local_path"
        ]
        .map(lambda path: path.is_file())
        .all()
    ),
}

print("Local methylation-payload checks:")
for check_name, check_passed in local_path_checks.items():
    print(f"{check_name}: {check_passed}")

print(
    "\nCandidate payloads available: "
    f"{methylation_candidate_inventory['methylation_local_path'].map(lambda path: path.is_file()).sum():,}"
)

if not all(local_path_checks.values()):
    raise FileNotFoundError(
        "One or more candidate methylation payloads are unavailable."
    )

Local methylation-payload checks:
local_paths_are_unique: True
all_candidate_payloads_are_available: True

Candidate payloads available: 10,038


In [10]:
# =============================================================================
# Inspect representative methylation payloads by platform
# =============================================================================

representative_payloads = (
    methylation_candidate_inventory
    .sort_values(
        [
            "methylation_platform",
            "methylation_file_id",
        ],
        kind="stable",
    )
    .groupby(
        "methylation_platform",
        sort=True,
    )
    .head(1)
)

for record in representative_payloads.itertuples(index=False):
    print("=" * 80)
    print(f"Platform: {record.methylation_platform}")
    print(
        "Payload:  "
        f"{project_relative_path(record.methylation_local_path)}"
    )
    print()

    with record.methylation_local_path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:
        for line_number in range(1, 6):
            line = handle.readline()

            if not line:
                break

            print(f"{line_number}: {line.rstrip()}")

Platform: Illumina Human Methylation 27
Payload:  data/raw/tcga/methylation/000bcedf-3094-409f-bc85-6570773877e5/be737c22-e2c5-41f0-9113-c6256541f151.methylation_array.sesame.level3betas.txt

1: cg00000292	0.936655975250212
2: cg00002426	0.521074693909812
3: cg00003994	0.0193747754649381
4: cg00005847	0.882645826423477
5: cg00006414	0.0248280575232252
Platform: Illumina Human Methylation 450
Payload:  data/raw/tcga/methylation/000f140d-c780-448d-804e-5ce533dbb5fb/e17e1d27-cbe1-4d21-af5e-786b80a3ef3e.methylation_array.sesame.level3betas.txt

1: cg00000029	0.429896459421591
2: cg00000108	0.961293876673234
3: cg00000109	0.933788927321535
4: cg00000165	0.447682403270816
5: cg00000236	0.928452277211974


In [11]:
# =============================================================================
# Define the canonical methylation-payload reader
# =============================================================================

METHYLATION_PAYLOAD_COLUMNS = [
    "probe_id",
    "beta_value",
]


def read_methylation_payload(file_path):
    payload = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=METHYLATION_PAYLOAD_COLUMNS,
        dtype="string",
        keep_default_na=False,
    )

    payload["probe_id"] = (
        payload["probe_id"]
        .str.strip()
    )

    beta_text = (
        payload["beta_value"]
        .str.strip()
    )

    missing_beta = (
        beta_text.eq("")
        | beta_text.str.upper().isin(
            ["NA", "NAN"]
        )
    )

    beta_numeric = pd.to_numeric(
        beta_text.mask(missing_beta),
        errors="coerce",
    )

    invalid_beta = (
        ~missing_beta
        & beta_numeric.isna()
    )

    if invalid_beta.any():
        raise ValueError(
            f"Non-numeric beta values detected in: {file_path}"
        )

    payload["beta_value"] = beta_numeric

    return payload

In [12]:
# =============================================================================
# Validate representative methylation payloads
# =============================================================================

representative_validation_records = []

for record in representative_payloads.itertuples(index=False):
    payload = read_methylation_payload(
        record.methylation_local_path
    )

    numeric_beta = payload["beta_value"].dropna()

    validation_record = {
        "platform": record.methylation_platform,
        "row_count": len(payload),
        "probe_ids_complete": payload["probe_id"].ne("").all(),
        "probe_ids_unique": payload["probe_id"].is_unique,
        "missing_beta_count": payload["beta_value"].isna().sum(),
        "beta_values_finite": np.isfinite(numeric_beta).all(),
        "beta_values_in_range": numeric_beta.between(0, 1).all(),
        "beta_min": numeric_beta.min(),
        "beta_max": numeric_beta.max(),
    }

    representative_validation_records.append(
        validation_record
    )

representative_payload_validation = pd.DataFrame(
    representative_validation_records
)

print("Representative methylation-payload validation:")
print(
    representative_payload_validation.to_string(
        index=False
    )
)

required_validation_columns = [
    "probe_ids_complete",
    "probe_ids_unique",
    "beta_values_finite",
    "beta_values_in_range",
]

if not representative_payload_validation[
    required_validation_columns
].all().all():
    raise ValueError(
        "One or more representative methylation payloads "
        "failed structural validation."
    )

Representative methylation-payload validation:
                      platform  row_count  probe_ids_complete  probe_ids_unique  missing_beta_count  beta_values_finite  beta_values_in_range  beta_min  beta_max
 Illumina Human Methylation 27      27578                True              True                2818                True                  True  0.003519  0.995429
Illumina Human Methylation 450     486427                True              True               66873                True                  True  0.007464  0.992686


In [13]:
# =============================================================================
# Establish platform-specific reference probe indices
# =============================================================================

platform_reference_probe_ids = {}
reference_probe_records = []

for record in representative_payloads.itertuples(index=False):
    payload = read_methylation_payload(
        record.methylation_local_path
    )

    probe_ids = pd.Index(
        payload["probe_id"],
        name="probe_id",
    )

    probe_prefixes = (
        payload["probe_id"]
        .str[:2]
    )

    platform_reference_probe_ids[
        record.methylation_platform
    ] = probe_ids

    reference_probe_records.append(
        {
            "platform": record.methylation_platform,
            "reference_file_id": record.methylation_file_id,
            "probe_count": len(probe_ids),
            "cg_probe_count": probe_prefixes.eq("cg").sum(),
            "rs_probe_count": probe_prefixes.eq("rs").sum(),
            "other_probe_count": (
                ~probe_prefixes.isin(["cg", "rs"])
            ).sum(),
        }
    )

reference_probe_summary = pd.DataFrame(
    reference_probe_records
)

reference_checks = {
    "all_candidate_platforms_have_reference": (
        set(
            methylation_candidate_inventory[
                "methylation_platform"
            ].unique()
        )
        == set(platform_reference_probe_ids)
    ),
    "reference_probe_ids_are_unique": all(
        probe_ids.is_unique
        for probe_ids in platform_reference_probe_ids.values()
    ),
}

print("Platform-specific reference probe indices established.")
print(
    reference_probe_summary.to_string(
        index=False
    )
)

print("\nReference-index checks:")
for check_name, check_passed in reference_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(reference_checks.values()):
    raise ValueError(
        "Platform-specific reference probe indices are invalid."
    )

Platform-specific reference probe indices established.
                      platform                    reference_file_id  probe_count  cg_probe_count  rs_probe_count  other_probe_count
 Illumina Human Methylation 27 000bcedf-3094-409f-bc85-6570773877e5        27578           27578               0                  0
Illumina Human Methylation 450 000f140d-c780-448d-804e-5ce533dbb5fb       486427          482421              65               3941

Reference-index checks:
all_candidate_platforms_have_reference: True
reference_probe_ids_are_unique: True


In [14]:
# =============================================================================
# Define per-file methylation-QC summarization
# =============================================================================

def summarize_methylation_payload(
    file_path,
    platform,
):
    payload = read_methylation_payload(file_path)

    reference_probe_ids = platform_reference_probe_ids[
        platform
    ]

    observed_probe_ids = pd.Index(
        payload["probe_id"],
        name="probe_id",
    )

    beta_values = payload["beta_value"]
    numeric_beta = beta_values.dropna().to_numpy()

    if numeric_beta.size:
        beta_quantiles = np.quantile(
            numeric_beta,
            [0.01, 0.05, 0.50, 0.95, 0.99],
        )
        beta_mean = numeric_beta.mean()
        beta_sd = numeric_beta.std(ddof=1)
        beta_min = numeric_beta.min()
        beta_max = numeric_beta.max()
    else:
        beta_quantiles = np.full(5, np.nan)
        beta_mean = np.nan
        beta_sd = np.nan
        beta_min = np.nan
        beta_max = np.nan

    return {
        "row_count": len(payload),
        "probe_ids_complete": (
            payload["probe_id"].notna().all()
            and payload["probe_id"].ne("").all()
        ),
        "probe_ids_unique": observed_probe_ids.is_unique,
        "probe_count_matches_reference": (
            len(observed_probe_ids)
            == len(reference_probe_ids)
        ),
        "probe_order_matches_reference": (
            observed_probe_ids.equals(reference_probe_ids)
        ),
        "missing_beta_count": beta_values.isna().sum(),
        "missing_beta_fraction": beta_values.isna().mean(),
        "numeric_beta_count": numeric_beta.size,
        "beta_values_finite": np.isfinite(numeric_beta).all(),
        "beta_below_zero_count": (numeric_beta < 0).sum(),
        "beta_above_one_count": (numeric_beta > 1).sum(),
        "beta_min": beta_min,
        "beta_q01": beta_quantiles[0],
        "beta_q05": beta_quantiles[1],
        "beta_median": beta_quantiles[2],
        "beta_mean": beta_mean,
        "beta_q95": beta_quantiles[3],
        "beta_q99": beta_quantiles[4],
        "beta_max": beta_max,
        "beta_sd": beta_sd,
    }

In [15]:
# =============================================================================
# Test per-file methylation-QC summarization
# =============================================================================

test_payload_inventory = (
    methylation_candidate_inventory
    .groupby(
        "methylation_platform",
        group_keys=False,
    )
    .sample(
        n=3,
        random_state=202,
    )
    .sort_values(
        [
            "methylation_platform",
            "methylation_file_id",
        ],
        kind="stable",
    )
)

test_qc_records = []

for record in test_payload_inventory.itertuples(index=False):
    qc_summary = summarize_methylation_payload(
        file_path=record.methylation_local_path,
        platform=record.methylation_platform,
    )

    test_qc_records.append(
        {
            "methylation_file_id": record.methylation_file_id,
            "methylation_platform": record.methylation_platform,
            **qc_summary,
        }
    )

test_methylation_qc = pd.DataFrame(test_qc_records)

test_structural_checks = {
    "all_probe_ids_complete": (
        test_methylation_qc["probe_ids_complete"].all()
    ),
    "all_probe_ids_unique": (
        test_methylation_qc["probe_ids_unique"].all()
    ),
    "all_probe_counts_match_reference": (
        test_methylation_qc[
            "probe_count_matches_reference"
        ].all()
    ),
    "all_probe_orders_match_reference": (
        test_methylation_qc[
            "probe_order_matches_reference"
        ].all()
    ),
    "all_numeric_betas_are_finite": (
        test_methylation_qc["beta_values_finite"].all()
    ),
    "all_beta_values_are_in_range": (
        test_methylation_qc[
            "beta_below_zero_count"
        ].eq(0).all()
        and test_methylation_qc[
            "beta_above_one_count"
        ].eq(0).all()
    ),
}

display_columns = [
    "methylation_file_id",
    "methylation_platform",
    "row_count",
    "missing_beta_count",
    "missing_beta_fraction",
    "beta_median",
    "beta_mean",
    "beta_sd",
]

print("Test methylation-QC summaries:")
print(
    test_methylation_qc[
        display_columns
    ].to_string(index=False)
)

print("\nTest structural checks:")
for check_name, check_passed in test_structural_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(test_structural_checks.values()):
    raise ValueError(
        "One or more test methylation payloads failed "
        "structural QC."
    )

Test methylation-QC summaries:
                 methylation_file_id           methylation_platform  row_count  missing_beta_count  missing_beta_fraction  beta_median  beta_mean  beta_sd
227be386-2b3b-4940-985f-8157f7678b02  Illumina Human Methylation 27      27578                2688               0.097469     0.035424   0.247491 0.344755
a9f397d2-e16f-4e8b-bcb9-26f029fab4a4  Illumina Human Methylation 27      27578                2622               0.095076     0.049260   0.261946 0.335089
d257bb06-d4f6-471b-a93e-9d1980d3def8  Illumina Human Methylation 27      27578                2640               0.095728     0.034631   0.279185 0.359644
2e355429-f6ab-41f9-8813-14c2c39192d9 Illumina Human Methylation 450     486427               67800               0.139384     0.543024   0.484395 0.359349
c82be059-3f95-4f9f-9e21-3209c5f31aeb Illumina Human Methylation 450     486427               82179               0.168944     0.457870   0.471159 0.364322
ff28d174-dc2b-4fd8-a1bc-886edc90669f Il

# =============================================================================
# Summarize all candidate methylation payloads
# =============================================================================

methylation_qc_records = []

total_payloads = len(methylation_candidate_inventory)
start_time = pd.Timestamp.now(tz="UTC")

for file_number, record in enumerate(
    methylation_candidate_inventory.itertuples(index=False),
    start=1,
):
    try:
        qc_summary = summarize_methylation_payload(
            file_path=record.methylation_local_path,
            platform=record.methylation_platform,
        )
    except Exception as error:
        raise RuntimeError(
            "Failed to summarize methylation payload "
            f"{record.methylation_file_id}: "
            f"{project_relative_path(record.methylation_local_path)}"
        ) from error

    methylation_qc_records.append(
        {
            "methylation_file_id": record.methylation_file_id,
            "methylation_platform": record.methylation_platform,
            **qc_summary,
        }
    )

    if (
        file_number % 250 == 0
        or file_number == total_payloads
    ):
        elapsed_seconds = (
            pd.Timestamp.now(tz="UTC") - start_time
        ).total_seconds()

        processing_rate = (
            file_number / elapsed_seconds
            if elapsed_seconds > 0
            else np.nan
        )

        remaining_seconds = (
            (total_payloads - file_number) / processing_rate
            if processing_rate > 0
            else np.nan
        )

        print(
            f"Processed {file_number:,}/{total_payloads:,} payloads "
            f"({processing_rate:,.2f} files/s; "
            f"ETA {remaining_seconds / 60:,.1f} min)"
        )

methylation_file_qc = pd.DataFrame(
    methylation_qc_records
)

print()
print("Candidate methylation payload summarization completed.")
print(f"QC records: {len(methylation_file_qc):,}")

# =============================================================================
# Persist per-file methylation-QC metrics
# =============================================================================

METHYLATION_FILE_QC_METRICS_PATH = (
    METHYLATION_QC_OUTPUT_DIR
    / (
        "tcga_primary_tumor_methylation_"
        "candidate_file_qc_metrics.csv"
    )
)

required_qc_metric_columns = [
    "methylation_file_id",
    "methylation_platform",
    "row_count",
    "probe_ids_complete",
    "probe_ids_unique",
    "probe_count_matches_reference",
    "probe_order_matches_reference",
    "missing_beta_count",
    "missing_beta_fraction",
    "numeric_beta_count",
    "beta_values_finite",
    "beta_below_zero_count",
    "beta_above_one_count",
    "beta_min",
    "beta_q01",
    "beta_q05",
    "beta_median",
    "beta_mean",
    "beta_q95",
    "beta_q99",
    "beta_max",
    "beta_sd",
]

missing_qc_metric_columns = [
    column
    for column in required_qc_metric_columns
    if column not in methylation_file_qc.columns
]

candidate_file_ids = (
    methylation_candidate_inventory[
        "methylation_file_id"
    ]
    .astype("string")
)

qc_file_ids = (
    methylation_file_qc[
        "methylation_file_id"
    ]
    .astype("string")
)

pre_write_checks = {
    "required_columns_are_present": (
        not missing_qc_metric_columns
    ),
    "qc_table_is_not_empty": (
        not methylation_file_qc.empty
    ),
    "qc_file_ids_are_complete": (
        qc_file_ids.notna().all()
    ),
    "qc_file_ids_are_unique": (
        qc_file_ids.is_unique
    ),
    "row_count_matches_candidate_inventory": (
        len(methylation_file_qc)
        == len(methylation_candidate_inventory)
    ),
    "file_ids_match_candidate_inventory": (
        set(qc_file_ids)
        == set(candidate_file_ids)
    ),
}

print("Methylation-QC metrics pre-write checks:")
for check_name, check_passed in pre_write_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(pre_write_checks.values()):
    raise ValueError(
        "Per-file methylation-QC metrics failed "
        "pre-write validation."
    )


# -----------------------------------------------------------------------------
# Atomic write
# -----------------------------------------------------------------------------

METHYLATION_QC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

temporary_output_path = (
    METHYLATION_FILE_QC_METRICS_PATH.with_name(
        METHYLATION_FILE_QC_METRICS_PATH.name + ".tmp"
    )
)

methylation_file_qc.to_csv(
    temporary_output_path,
    index=False,
)

written_methylation_file_qc = pd.read_csv(
    temporary_output_path,
    low_memory=False,
)

written_checks = {
    "temporary_output_exists": (
        temporary_output_path.is_file()
    ),
    "written_shape_matches_memory": (
        written_methylation_file_qc.shape
        == methylation_file_qc.shape
    ),
    "written_columns_match_memory": (
        written_methylation_file_qc.columns.tolist()
        == methylation_file_qc.columns.tolist()
    ),
    "written_file_identity_and_order_match_memory": (
        written_methylation_file_qc[
            "methylation_file_id"
        ].astype("string").tolist()
        == qc_file_ids.tolist()
    ),
}

print("\nWritten-artifact checks:")
for check_name, check_passed in written_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(written_checks.values()):
    temporary_output_path.unlink(missing_ok=True)

    raise IOError(
        "Written methylation-QC metrics failed "
        "round-trip validation."
    )

temporary_output_path.replace(
    METHYLATION_FILE_QC_METRICS_PATH
)

print("\nPer-file methylation-QC metrics saved.")
print(
    "Path: "
    f"{project_relative_path(METHYLATION_FILE_QC_METRICS_PATH)}"
)
print(
    f"Shape: {methylation_file_qc.shape}"
)
print(
    "File size: "
    f"{METHYLATION_FILE_QC_METRICS_PATH.stat().st_size / (1024**2):,.3f} MiB"
)

In [16]:
# =============================================================================
# Load persisted per-file methylation-QC metrics
# =============================================================================

METHYLATION_FILE_QC_METRICS_PATH = (
    METHYLATION_QC_OUTPUT_DIR
    / (
        "tcga_primary_tumor_methylation_"
        "candidate_file_qc_metrics.csv"
    )
)

if not METHYLATION_FILE_QC_METRICS_PATH.is_file():
    raise FileNotFoundError(
        "Persisted methylation-QC metrics were not found: "
        f"{METHYLATION_FILE_QC_METRICS_PATH}"
    )

methylation_file_qc = pd.read_csv(
    METHYLATION_FILE_QC_METRICS_PATH,
    dtype={
        "methylation_file_id": "string",
        "methylation_platform": "string",
    },
    low_memory=False,
)


# -----------------------------------------------------------------------------
# Restore and validate logical columns
# -----------------------------------------------------------------------------

boolean_qc_columns = [
    "probe_ids_complete",
    "probe_ids_unique",
    "probe_count_matches_reference",
    "probe_order_matches_reference",
    "beta_values_finite",
]

for column in boolean_qc_columns:
    methylation_file_qc[column] = (
        methylation_file_qc[column]
        .astype("boolean")
    )

candidate_file_ids = set(
    methylation_candidate_inventory[
        "methylation_file_id"
    ].astype("string")
)

loaded_file_ids = set(
    methylation_file_qc[
        "methylation_file_id"
    ]
)

load_checks = {
    "qc_table_is_not_empty": (
        not methylation_file_qc.empty
    ),
    "file_ids_are_complete": (
        methylation_file_qc[
            "methylation_file_id"
        ].notna().all()
    ),
    "file_ids_are_unique": (
        methylation_file_qc[
            "methylation_file_id"
        ].is_unique
    ),
    "row_count_matches_candidate_inventory": (
        len(methylation_file_qc)
        == len(methylation_candidate_inventory)
    ),
    "file_ids_match_candidate_inventory": (
        loaded_file_ids == candidate_file_ids
    ),
}

print("Persisted methylation-QC metrics loaded.")
print(
    f"Path:  "
    f"{project_relative_path(METHYLATION_FILE_QC_METRICS_PATH)}"
)
print(f"Shape: {methylation_file_qc.shape}")

print("\nLoaded-artifact checks:")
for check_name, check_passed in load_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(load_checks.values()):
    raise ValueError(
        "Persisted methylation-QC metrics do not match "
        "the current candidate-file inventory."
    )

Persisted methylation-QC metrics loaded.
Path:  data/interim/qc/tcga_primary_tumor_methylation_candidate_file_qc_metrics.csv
Shape: (10038, 22)

Loaded-artifact checks:
qc_table_is_not_empty: True
file_ids_are_complete: True
file_ids_are_unique: True
row_count_matches_candidate_inventory: True
file_ids_match_candidate_inventory: True


In [17]:
# =============================================================================
# Validate persisted per-file methylation-QC metrics
# =============================================================================

expected_probe_counts = {
    record.platform: record.probe_count
    for record in reference_probe_summary.itertuples(index=False)
}

expected_row_count = (
    methylation_file_qc["methylation_platform"]
    .map(expected_probe_counts)
)

quantile_columns = [
    "beta_min",
    "beta_q01",
    "beta_q05",
    "beta_median",
    "beta_q95",
    "beta_q99",
    "beta_max",
]

quantile_order_is_valid = (
    methylation_file_qc[quantile_columns]
    .diff(axis=1)
    .iloc[:, 1:]
    .ge(0)
    .all(axis=1)
)

global_qc_metric_checks = {
    "all_platforms_are_expected": (
        methylation_file_qc["methylation_platform"]
        .isin(expected_probe_counts)
        .all()
    ),
    "all_row_counts_match_platform_reference": (
        methylation_file_qc["row_count"]
        .eq(expected_row_count)
        .all()
    ),
    "all_probe_ids_are_complete": (
        methylation_file_qc["probe_ids_complete"].all()
    ),
    "all_probe_ids_are_unique": (
        methylation_file_qc["probe_ids_unique"].all()
    ),
    "all_probe_counts_match_reference": (
        methylation_file_qc[
            "probe_count_matches_reference"
        ].all()
    ),
    "all_probe_orders_match_reference": (
        methylation_file_qc[
            "probe_order_matches_reference"
        ].all()
    ),
    "numeric_and_missing_counts_reconstruct_rows": (
        (
            methylation_file_qc["numeric_beta_count"]
            + methylation_file_qc["missing_beta_count"]
        )
        .eq(methylation_file_qc["row_count"])
        .all()
    ),
    "all_numeric_beta_values_are_finite": (
        methylation_file_qc["beta_values_finite"].all()
    ),
    "no_beta_values_are_below_zero": (
        methylation_file_qc[
            "beta_below_zero_count"
        ].eq(0).all()
    ),
    "no_beta_values_are_above_one": (
        methylation_file_qc[
            "beta_above_one_count"
        ].eq(0).all()
    ),
    "all_beta_quantiles_are_ordered": (
        quantile_order_is_valid.all()
    ),
}

print("Persisted methylation-QC metric checks:")
for check_name, check_passed in global_qc_metric_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(global_qc_metric_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in global_qc_metric_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Persisted methylation-QC metrics failed validation: "
        + ", ".join(failed_checks)
    )

Persisted methylation-QC metric checks:
all_platforms_are_expected: True
all_row_counts_match_platform_reference: True
all_probe_ids_are_complete: True
all_probe_ids_are_unique: True
all_probe_counts_match_reference: True
all_probe_orders_match_reference: True
numeric_and_missing_counts_reconstruct_rows: True
all_numeric_beta_values_are_finite: True
no_beta_values_are_below_zero: True
no_beta_values_are_above_one: True
all_beta_quantiles_are_ordered: True


In [18]:
# =============================================================================
# Summarize platform-specific sample-level QC distributions
# =============================================================================

sample_qc_metrics = [
    "missing_beta_fraction",
    "beta_median",
    "beta_mean",
    "beta_sd",
]

summary_quantiles = [
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    1.00,
]

platform_file_counts = (
    methylation_file_qc[
        "methylation_platform"
    ]
    .value_counts()
    .sort_index()
    .rename("file_count")
)

platform_qc_distribution = (
    methylation_file_qc
    .groupby(
        "methylation_platform",
        sort=True,
    )[sample_qc_metrics]
    .quantile(summary_quantiles)
    .rename_axis(
        index=[
            "methylation_platform",
            "quantile",
        ]
    )
    .reset_index()
)

platform_qc_distribution["quantile"] = (
    platform_qc_distribution["quantile"]
    .map(
        lambda value: (
            f"p{int(round(value * 100)):02d}"
        )
    )
)

print("Candidate methylation files by platform:")
print(platform_file_counts)

print("\nPlatform-specific sample-level QC distributions:")
print(
    platform_qc_distribution.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

Candidate methylation files by platform:
methylation_platform
Illumina Human Methylation 27     1621
Illumina Human Methylation 450    8417
Name: file_count, dtype: int64[pyarrow]

Platform-specific sample-level QC distributions:
          methylation_platform quantile  missing_beta_fraction  beta_median  beta_mean  beta_sd
 Illumina Human Methylation 27      p00               0.083219     0.018918   0.151118 0.230516
 Illumina Human Methylation 27      p01               0.085989     0.022337   0.183342 0.272797
 Illumina Human Methylation 27      p05               0.088694     0.025722   0.203203 0.295364
 Illumina Human Methylation 27      p25               0.095185     0.036431   0.232877 0.315979
 Illumina Human Methylation 27      p50               0.102038     0.046732   0.250260 0.328250
 Illumina Human Methylation 27      p75               0.111792     0.057943   0.266610 0.339966
 Illumina Human Methylation 27      p95               0.139133     0.074869   0.297486 0.358459
 I

In [19]:
# =============================================================================
# Attach cohort metadata to per-file methylation-QC metrics
# =============================================================================

methylation_qc_metadata_columns = [
    "methylation_file_id",
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "methylation_aliquot_submitter_id",
    "methylation_platform",
    "eligible_candidate_pair_count",
]

methylation_file_qc_annotated = (
    methylation_file_qc
    .merge(
        methylation_candidate_inventory[
            methylation_qc_metadata_columns
        ],
        on=[
            "methylation_file_id",
            "methylation_platform",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

annotation_checks = {
    "all_qc_records_match_candidate_metadata": (
        methylation_file_qc_annotated["_merge"]
        .eq("both")
        .all()
    ),
    "project_ids_are_complete": (
        methylation_file_qc_annotated[
            "project_id"
        ].notna().all()
    ),
    "case_ids_are_complete": (
        methylation_file_qc_annotated[
            "case_submitter_id"
        ].notna().all()
    ),
    "platforms_remain_complete": (
        methylation_file_qc_annotated[
            "methylation_platform"
        ].notna().all()
    ),
}

methylation_file_qc_annotated = (
    methylation_file_qc_annotated
    .drop(columns="_merge")
)

print("Methylation-QC metadata annotation checks:")
for check_name, check_passed in annotation_checks.items():
    print(f"{check_name}: {check_passed}")

print(
    "\nAnnotated QC table shape: "
    f"{methylation_file_qc_annotated.shape}"
)
print(
    "Projects represented:     "
    f"{methylation_file_qc_annotated['project_id'].nunique():,}"
)

if not all(annotation_checks.values()):
    raise ValueError(
        "Per-file methylation-QC metrics could not be "
        "fully matched to candidate-file metadata."
    )

Methylation-QC metadata annotation checks:
all_qc_records_match_candidate_metadata: True
project_ids_are_complete: True
case_ids_are_complete: True
platforms_remain_complete: True

Annotated QC table shape: (10038, 27)
Projects represented:     33


In [20]:
# =============================================================================
# Characterize project-by-platform cohort sizes
# =============================================================================

project_platform_counts = (
    methylation_file_qc_annotated
    .groupby(
        [
            "methylation_platform",
            "project_id",
        ],
        observed=True,
    )
    .size()
    .rename("file_count")
    .reset_index()
    .sort_values(
        [
            "methylation_platform",
            "file_count",
            "project_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

platform_stratum_summary = (
    project_platform_counts
    .groupby(
        "methylation_platform",
        observed=True,
    )["file_count"]
    .agg(
        project_count="size",
        minimum_files="min",
        median_files="median",
        maximum_files="max",
        total_files="sum",
    )
    .reset_index()
)

print("Project-by-platform cohort-size summary:")
print(
    platform_stratum_summary.to_string(
        index=False,
    )
)

print("\nTen smallest project-by-platform strata:")
print(
    project_platform_counts
    .head(10)
    .to_string(index=False)
)

print("\nProjects represented by platform:")
print(
    project_platform_counts
    .groupby(
        "methylation_platform",
        observed=True,
    )["project_id"]
    .nunique()
)

Project-by-platform cohort-size summary:
          methylation_platform  project_count  minimum_files  median_files  maximum_files  total_files
 Illumina Human Methylation 27             11              6         131.0            413         1621
Illumina Human Methylation 450             33              9         184.0            788         8417

Ten smallest project-by-platform strata:
         methylation_platform project_id  file_count
Illumina Human Methylation 27  TCGA-KIRP           6
Illumina Human Methylation 27  TCGA-STAD          39
Illumina Human Methylation 27  TCGA-LUAD          59
Illumina Human Methylation 27  TCGA-READ          67
Illumina Human Methylation 27  TCGA-UCEC         115
Illumina Human Methylation 27  TCGA-LUSC         131
Illumina Human Methylation 27   TCGA-GBM         150
Illumina Human Methylation 27  TCGA-COAD         160
Illumina Human Methylation 27  TCGA-KIRC         169
Illumina Human Methylation 27  TCGA-BRCA         312

Projects represented by 

In [21]:
# =============================================================================
# Characterize methylation-file multiplicity by case
# =============================================================================

case_methylation_multiplicity = (
    methylation_file_qc_annotated
    .groupby(
        [
            "project_id",
            "case_submitter_id",
        ],
        observed=True,
    )
    .agg(
        methylation_file_count=(
            "methylation_file_id",
            "nunique",
        ),
        methylation_platform_count=(
            "methylation_platform",
            "nunique",
        ),
    )
    .reset_index()
)

case_platform_multiplicity = (
    methylation_file_qc_annotated
    .groupby(
        [
            "project_id",
            "case_submitter_id",
            "methylation_platform",
        ],
        observed=True,
    )
    .size()
    .rename("methylation_file_count")
    .reset_index()
)

multiplicity_checks = {
    "case_count_matches_candidate_inventory": (
        len(case_methylation_multiplicity)
        == methylation_candidate_inventory[
            "case_submitter_id"
        ].nunique()
    ),
    "file_counts_reconstruct_candidate_inventory": (
        case_methylation_multiplicity[
            "methylation_file_count"
        ].sum()
        == len(methylation_file_qc_annotated)
    ),
}

print("Case-level methylation-file multiplicity:")
print(
    case_methylation_multiplicity[
        "methylation_file_count"
    ]
    .value_counts()
    .sort_index()
)

print("\nPlatforms represented per case:")
print(
    case_methylation_multiplicity[
        "methylation_platform_count"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nCases with multiple files on the same platform: "
    f"{case_platform_multiplicity['methylation_file_count'].gt(1).sum():,}"
)

print("\nMultiplicity checks:")
for check_name, check_passed in multiplicity_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(multiplicity_checks.values()):
    raise ValueError(
        "Case-level methylation-file multiplicity is inconsistent."
    )

Case-level methylation-file multiplicity:
methylation_file_count
1    9928
2      25
3      20
Name: count, dtype: int64

Platforms represented per case:
methylation_platform_count
1    9971
2       2
Name: count, dtype: int64

Cases with multiple files on the same platform: 43

Multiplicity checks:
case_count_matches_candidate_inventory: True
file_counts_reconstruct_candidate_inventory: True


In [22]:
# =============================================================================
# Characterize the origin of within-case methylation multiplicity
# =============================================================================

multi_file_cases = (
    case_methylation_multiplicity.loc[
        case_methylation_multiplicity[
            "methylation_file_count"
        ].gt(1),
        [
            "project_id",
            "case_submitter_id",
        ],
    ]
)

multi_file_case_records = (
    methylation_file_qc_annotated
    .merge(
        multi_file_cases,
        on=[
            "project_id",
            "case_submitter_id",
        ],
        how="inner",
        validate="many_to_one",
    )
)

multi_file_case_structure = (
    multi_file_case_records
    .groupby(
        [
            "project_id",
            "case_submitter_id",
        ],
        observed=True,
    )
    .agg(
        file_count=(
            "methylation_file_id",
            "nunique",
        ),
        sample_count=(
            "sample_submitter_id",
            "nunique",
        ),
        aliquot_count=(
            "methylation_aliquot_submitter_id",
            "nunique",
        ),
        platform_count=(
            "methylation_platform",
            "nunique",
        ),
    )
    .reset_index()
)

multi_file_case_structure["multiplicity_structure"] = np.select(
    [
        multi_file_case_structure["aliquot_count"].eq(1),
        (
            multi_file_case_structure["sample_count"].eq(1)
            & multi_file_case_structure["aliquot_count"].gt(1)
        ),
        multi_file_case_structure["sample_count"].gt(1),
    ],
    [
        "Multiple files from the same aliquot",
        "Multiple aliquots from the same sample",
        "Multiple samples from the same case",
    ],
    default="Unresolved multiplicity structure",
)

structure_checks = {
    "all_multi_file_cases_are_represented": (
        len(multi_file_case_structure)
        == multi_file_cases.shape[0]
    ),
    "biospecimen_identifiers_are_complete": (
        multi_file_case_records[
            [
                "sample_submitter_id",
                "methylation_aliquot_submitter_id",
            ]
        ]
        .notna()
        .all()
        .all()
    ),
    "all_structures_are_resolved": (
        multi_file_case_structure[
            "multiplicity_structure"
        ]
        .ne("Unresolved multiplicity structure")
        .all()
    ),
}

print(
    "Cases with multiple methylation files: "
    f"{len(multi_file_case_structure):,}"
)

print("\nMultiplicity structure:")
print(
    multi_file_case_structure[
        "multiplicity_structure"
    ]
    .value_counts()
)

print("\nPlatform count among multi-file cases:")
print(
    multi_file_case_structure[
        "platform_count"
    ]
    .value_counts()
    .sort_index()
)

print("\nMultiplicity-structure checks:")
for check_name, check_passed in structure_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(structure_checks.values()):
    raise ValueError(
        "Within-case methylation multiplicity could not be "
        "resolved consistently."
    )

Cases with multiple methylation files: 45

Multiplicity structure:
multiplicity_structure
Multiple samples from the same case       44
Multiple aliquots from the same sample     1
Name: count, dtype: int64

Platform count among multi-file cases:
platform_count
1    43
2     2
Name: count, dtype: int64

Multiplicity-structure checks:
all_multi_file_cases_are_represented: True
biospecimen_identifiers_are_complete: True
all_structures_are_resolved: True


In [23]:
# =============================================================================
# Identify eligible within-sample methylation comparators
# =============================================================================

within_sample_comparator_groups = (
    methylation_file_qc_annotated
    .groupby(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "methylation_platform",
        ],
        observed=True,
    )
    .agg(
        file_count=(
            "methylation_file_id",
            "nunique",
        ),
        aliquot_count=(
            "methylation_aliquot_submitter_id",
            "nunique",
        ),
    )
    .reset_index()
)

within_sample_comparator_groups = (
    within_sample_comparator_groups.loc[
        within_sample_comparator_groups[
            "file_count"
        ].gt(1)
    ]
    .reset_index(drop=True)
)

within_sample_comparator_inventory = (
    methylation_file_qc_annotated
    .merge(
        within_sample_comparator_groups[
            [
                "project_id",
                "case_submitter_id",
                "sample_submitter_id",
                "methylation_platform",
            ]
        ],
        on=[
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "methylation_platform",
        ],
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        [
            "project_id",
            "case_submitter_id",
            "sample_submitter_id",
            "methylation_file_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

comparator_checks = {
    "comparator_groups_are_not_cross_platform": (
        within_sample_comparator_inventory
        .groupby(
            [
                "project_id",
                "case_submitter_id",
                "sample_submitter_id",
            ],
            observed=True,
        )["methylation_platform"]
        .nunique()
        .eq(1)
        .all()
    ),
    "each_group_has_multiple_files": (
        within_sample_comparator_groups[
            "file_count"
        ].gt(1).all()
    ),
    "all_comparator_records_are_complete": (
        within_sample_comparator_inventory[
            [
                "methylation_file_id",
                "methylation_aliquot_submitter_id",
                "missing_beta_fraction",
                "beta_median",
                "beta_mean",
                "beta_sd",
            ]
        ]
        .notna()
        .all()
        .all()
    ),
}

print(
    "Eligible within-sample comparator groups: "
    f"{len(within_sample_comparator_groups):,}"
)
print(
    "Files represented: "
    f"{len(within_sample_comparator_inventory):,}"
)

if not within_sample_comparator_inventory.empty:
    print("\nWithin-sample comparator inventory:")
    print(
        within_sample_comparator_inventory[
            [
                "project_id",
                "case_submitter_id",
                "sample_submitter_id",
                "methylation_aliquot_submitter_id",
                "methylation_file_id",
                "methylation_platform",
                "missing_beta_fraction",
                "beta_median",
                "beta_mean",
                "beta_sd",
            ]
        ].to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}",
        )
    )

print("\nComparator checks:")
for check_name, check_passed in comparator_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(comparator_checks.values()):
    raise ValueError(
        "Within-sample methylation comparator inventory "
        "is inconsistent."
    )

Eligible within-sample comparator groups: 21
Files represented: 42

Within-sample comparator inventory:
project_id case_submitter_id sample_submitter_id methylation_aliquot_submitter_id                  methylation_file_id           methylation_platform  missing_beta_fraction  beta_median  beta_mean  beta_sd
 TCGA-BLCA      TCGA-BL-A0C8    TCGA-BL-A0C8-01A     TCGA-BL-A0C8-01A-11D-A276-05 2970e797-fff4-4420-85ce-b316d2d9613f Illumina Human Methylation 450               0.159681     0.462922   0.465856 0.343345
 TCGA-BLCA      TCGA-BL-A0C8    TCGA-BL-A0C8-01A     TCGA-BL-A0C8-01A-11D-A10W-05 632a807c-c2f4-4385-bbfd-65c70b6674ee Illumina Human Methylation 450               0.192833     0.491666   0.480360 0.352272
 TCGA-BLCA      TCGA-BL-A13I    TCGA-BL-A13I-01A     TCGA-BL-A13I-01A-11D-A276-05 3900f6d7-2ae9-43c0-9e82-5dc4433bbea3 Illumina Human Methylation 450               0.162304     0.522460   0.484365 0.352534
 TCGA-BLCA      TCGA-BL-A13I    TCGA-BL-A13I-01A     TCGA-BL-A13I-01A-11

In [24]:
# =============================================================================
# Quantify within-sample methylation-QC discordance
# =============================================================================

comparator_group_columns = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "methylation_platform",
]

within_sample_comparator_summary = (
    within_sample_comparator_inventory
    .groupby(
        comparator_group_columns,
        observed=True,
    )
    .agg(
        file_count=(
            "methylation_file_id",
            "nunique",
        ),
        aliquot_count=(
            "methylation_aliquot_submitter_id",
            "nunique",
        ),
        missing_beta_fraction_min=(
            "missing_beta_fraction",
            "min",
        ),
        missing_beta_fraction_max=(
            "missing_beta_fraction",
            "max",
        ),
        beta_median_min=(
            "beta_median",
            "min",
        ),
        beta_median_max=(
            "beta_median",
            "max",
        ),
        beta_mean_min=(
            "beta_mean",
            "min",
        ),
        beta_mean_max=(
            "beta_mean",
            "max",
        ),
        beta_sd_min=(
            "beta_sd",
            "min",
        ),
        beta_sd_max=(
            "beta_sd",
            "max",
        ),
    )
    .reset_index()
)

within_sample_comparator_summary[
    "missing_beta_fraction_delta"
] = (
    within_sample_comparator_summary[
        "missing_beta_fraction_max"
    ]
    - within_sample_comparator_summary[
        "missing_beta_fraction_min"
    ]
)

within_sample_comparator_summary[
    "beta_median_delta"
] = (
    within_sample_comparator_summary["beta_median_max"]
    - within_sample_comparator_summary["beta_median_min"]
)

within_sample_comparator_summary[
    "beta_mean_delta"
] = (
    within_sample_comparator_summary["beta_mean_max"]
    - within_sample_comparator_summary["beta_mean_min"]
)

within_sample_comparator_summary[
    "beta_sd_delta"
] = (
    within_sample_comparator_summary["beta_sd_max"]
    - within_sample_comparator_summary["beta_sd_min"]
)

comparator_summary_checks = {
    "all_groups_have_two_files": (
        within_sample_comparator_summary[
            "file_count"
        ].eq(2).all()
    ),
    "all_groups_have_distinct_aliquots": (
        within_sample_comparator_summary[
            "aliquot_count"
        ].eq(2).all()
    ),
    "all_deltas_are_nonnegative": (
        within_sample_comparator_summary[
            [
                "missing_beta_fraction_delta",
                "beta_median_delta",
                "beta_mean_delta",
                "beta_sd_delta",
            ]
        ]
        .ge(0)
        .all()
        .all()
    ),
}

print("Within-sample comparator discordance summary:")
print(
    within_sample_comparator_summary[
        [
            "project_id",
            "case_submitter_id",
            "methylation_platform",
            "missing_beta_fraction_delta",
            "beta_median_delta",
            "beta_mean_delta",
            "beta_sd_delta",
        ]
    ]
    .sort_values(
        "missing_beta_fraction_delta",
        ascending=False,
        kind="stable",
    )
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nComparator-summary checks:")
for check_name, check_passed in comparator_summary_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(comparator_summary_checks.values()):
    raise ValueError(
        "Within-sample methylation comparator summaries "
        "are inconsistent."
    )

Within-sample comparator discordance summary:
project_id case_submitter_id           methylation_platform  missing_beta_fraction_delta  beta_median_delta  beta_mean_delta  beta_sd_delta
 TCGA-UCEC      TCGA-BK-A139 Illumina Human Methylation 450                     0.049407           0.004768         0.000832       0.013855
 TCGA-UCEC      TCGA-BK-A0CA Illumina Human Methylation 450                     0.037319           0.004311         0.002704       0.019956
 TCGA-COAD      TCGA-A6-5659 Illumina Human Methylation 450                     0.036044           0.009571         0.005158       0.000015
 TCGA-BLCA      TCGA-BL-A0C8 Illumina Human Methylation 450                     0.033152           0.028743         0.014504       0.008926
 TCGA-LUSC      TCGA-21-1076  Illumina Human Methylation 27                     0.019871           0.004164         0.009886       0.010186
 TCGA-BLCA      TCGA-BL-A13I Illumina Human Methylation 450                     0.017100           0.013332       

In [25]:
# =============================================================================
# Define platform-aware QC reference strata
# =============================================================================

MIN_PROJECT_PLATFORM_REFERENCE_SIZE = 30

project_platform_reference_sizes = (
    methylation_file_qc_annotated
    .groupby(
        [
            "methylation_platform",
            "project_id",
        ],
        observed=True,
    )
    .size()
    .rename("project_platform_file_count")
    .reset_index()
)

methylation_file_qc_annotated = (
    methylation_file_qc_annotated
    .merge(
        project_platform_reference_sizes,
        on=[
            "methylation_platform",
            "project_id",
        ],
        how="left",
        validate="many_to_one",
    )
)

methylation_file_qc_annotated[
    "qc_reference_level"
] = np.where(
    methylation_file_qc_annotated[
        "project_platform_file_count"
    ].ge(MIN_PROJECT_PLATFORM_REFERENCE_SIZE),
    "project_platform",
    "platform",
)

methylation_file_qc_annotated[
    "qc_reference_group"
] = np.where(
    methylation_file_qc_annotated[
        "qc_reference_level"
    ].eq("project_platform"),
    (
        methylation_file_qc_annotated[
            "methylation_platform"
        ].astype(str)
        + " | "
        + methylation_file_qc_annotated[
            "project_id"
        ].astype(str)
    ),
    methylation_file_qc_annotated[
        "methylation_platform"
    ].astype(str),
)

reference_assignment_checks = {
    "reference_sizes_are_complete": (
        methylation_file_qc_annotated[
            "project_platform_file_count"
        ].notna().all()
    ),
    "reference_levels_are_complete": (
        methylation_file_qc_annotated[
            "qc_reference_level"
        ].notna().all()
    ),
    "small_strata_use_platform_reference": (
        methylation_file_qc_annotated.loc[
            methylation_file_qc_annotated[
                "project_platform_file_count"
            ].lt(MIN_PROJECT_PLATFORM_REFERENCE_SIZE),
            "qc_reference_level",
        ]
        .eq("platform")
        .all()
    ),
}

print("QC reference assignments:")
print(
    methylation_file_qc_annotated[
        "qc_reference_level"
    ].value_counts()
)

print("\nProject-platform strata using platform fallback:")
print(
    project_platform_reference_sizes.loc[
        project_platform_reference_sizes[
            "project_platform_file_count"
        ].lt(MIN_PROJECT_PLATFORM_REFERENCE_SIZE)
    ]
    .sort_values(
        [
            "methylation_platform",
            "project_platform_file_count",
            "project_id",
        ],
        kind="stable",
    )
    .to_string(index=False)
)

print("\nReference-assignment checks:")
for check_name, check_passed in reference_assignment_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(reference_assignment_checks.values()):
    raise ValueError(
        "Platform-aware QC reference assignment is inconsistent."
    )

QC reference assignments:
qc_reference_level
project_platform    10023
platform               15
Name: count, dtype: int64

Project-platform strata using platform fallback:
          methylation_platform project_id  project_platform_file_count
 Illumina Human Methylation 27  TCGA-KIRP                            6
Illumina Human Methylation 450    TCGA-OV                            9

Reference-assignment checks:
reference_sizes_are_complete: True
reference_levels_are_complete: True
small_strata_use_platform_reference: True


In [26]:
# =============================================================================
# Compute robust QC reference statistics
# =============================================================================

QC_REFERENCE_METRICS = [
    "missing_beta_fraction",
    "beta_median",
    "beta_mean",
    "beta_sd",
]


def summarize_robust_reference(group):
    summary = {
        "reference_file_count": len(group),
    }

    for metric in QC_REFERENCE_METRICS:
        values = (
            pd.to_numeric(
                group[metric],
                errors="coerce",
            )
            .dropna()
            .to_numpy(dtype=float)
        )

        median = np.median(values)
        mad = np.median(
            np.abs(values - median)
        )

        summary[f"{metric}_reference_median"] = median
        summary[f"{metric}_reference_mad"] = mad

    return pd.Series(summary)


# -----------------------------------------------------------------------------
# Platform-wide reference statistics
# -----------------------------------------------------------------------------

platform_reference_statistics = (
    methylation_file_qc_annotated
    .groupby(
        "methylation_platform",
        observed=True,
    )
    .apply(
        summarize_robust_reference,
        include_groups=False,
    )
    .reset_index()
)


# -----------------------------------------------------------------------------
# Project-by-platform reference statistics
# -----------------------------------------------------------------------------

eligible_project_platform_records = (
    methylation_file_qc_annotated.loc[
        methylation_file_qc_annotated[
            "project_platform_file_count"
        ].ge(MIN_PROJECT_PLATFORM_REFERENCE_SIZE)
    ]
)

project_platform_reference_statistics = (
    eligible_project_platform_records
    .groupby(
        [
            "methylation_platform",
            "project_id",
        ],
        observed=True,
    )
    .apply(
        summarize_robust_reference,
        include_groups=False,
    )
    .reset_index()
)


# -----------------------------------------------------------------------------
# Basic integrity checks
# -----------------------------------------------------------------------------

mad_columns = [
    f"{metric}_reference_mad"
    for metric in QC_REFERENCE_METRICS
]

reference_statistics_checks = {
    "platform_references_are_complete": (
        platform_reference_statistics
        .notna()
        .all()
        .all()
    ),
    "project_platform_references_are_complete": (
        project_platform_reference_statistics
        .notna()
        .all()
        .all()
    ),
    "platform_mads_are_positive": (
        platform_reference_statistics[
            mad_columns
        ]
        .gt(0)
        .all()
        .all()
    ),
    "project_platform_mads_are_positive": (
        project_platform_reference_statistics[
            mad_columns
        ]
        .gt(0)
        .all()
        .all()
    ),
}

print("Robust QC reference statistics computed.")
print(
    "Platform reference groups:         "
    f"{len(platform_reference_statistics):,}"
)
print(
    "Project-platform reference groups: "
    f"{len(project_platform_reference_statistics):,}"
)

print("\nReference-statistics checks:")
for check_name, check_passed in reference_statistics_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(reference_statistics_checks.values()):
    raise ValueError(
        "Robust methylation-QC reference statistics "
        "are incomplete or degenerate."
    )

Robust QC reference statistics computed.
Platform reference groups:         2
Project-platform reference groups: 42

Reference-statistics checks:
platform_references_are_complete: True
project_platform_references_are_complete: True
platform_mads_are_positive: True
project_platform_mads_are_positive: True


In [27]:
# =============================================================================
# Assign robust reference statistics and compute standardized deviations
# =============================================================================

MODIFIED_Z_SCALE = 0.6744897501960817


# -----------------------------------------------------------------------------
# Build unified reference table
# -----------------------------------------------------------------------------

platform_reference_table = (
    platform_reference_statistics
    .assign(
        qc_reference_level="platform",
        qc_reference_group=lambda table: (
            table["methylation_platform"].astype(str)
        ),
    )
)

project_platform_reference_table = (
    project_platform_reference_statistics
    .assign(
        qc_reference_level="project_platform",
        qc_reference_group=lambda table: (
            table["methylation_platform"].astype(str)
            + " | "
            + table["project_id"].astype(str)
        ),
    )
)

robust_reference_table = pd.concat(
    [
        platform_reference_table,
        project_platform_reference_table,
    ],
    ignore_index=True,
    sort=False,
)

reference_value_columns = [
    "reference_file_count",
    *[
        f"{metric}_reference_{statistic}"
        for metric in QC_REFERENCE_METRICS
        for statistic in ["median", "mad"]
    ],
]

robust_reference_table = robust_reference_table[
    [
        "qc_reference_level",
        "qc_reference_group",
        *reference_value_columns,
    ]
]


# -----------------------------------------------------------------------------
# Attach selected references to each candidate file
# -----------------------------------------------------------------------------

methylation_file_qc_annotated = (
    methylation_file_qc_annotated
    .merge(
        robust_reference_table,
        on=[
            "qc_reference_level",
            "qc_reference_group",
        ],
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

reference_assignment_checks = {
    "reference_groups_are_unique": (
        robust_reference_table[
            [
                "qc_reference_level",
                "qc_reference_group",
            ]
        ]
        .duplicated()
        .sum()
        == 0
    ),
    "all_files_receive_a_reference": (
        methylation_file_qc_annotated["_merge"]
        .eq("both")
        .all()
    ),
    "reference_statistics_are_complete": (
        methylation_file_qc_annotated[
            reference_value_columns
        ]
        .notna()
        .all()
        .all()
    ),
}

methylation_file_qc_annotated = (
    methylation_file_qc_annotated
    .drop(columns="_merge")
)


# -----------------------------------------------------------------------------
# Compute signed modified z-scores
# -----------------------------------------------------------------------------

for metric in QC_REFERENCE_METRICS:
    methylation_file_qc_annotated[
        f"{metric}_modified_z"
    ] = (
        MODIFIED_Z_SCALE
        * (
            methylation_file_qc_annotated[metric]
            - methylation_file_qc_annotated[
                f"{metric}_reference_median"
            ]
        )
        / methylation_file_qc_annotated[
            f"{metric}_reference_mad"
        ]
    )

print("Robust reference assignment and scoring completed.")
print(
    f"Candidate files scored: "
    f"{len(methylation_file_qc_annotated):,}"
)

print("\nReference-assignment checks:")
for check_name, check_passed in reference_assignment_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(reference_assignment_checks.values()):
    raise ValueError(
        "Robust QC references could not be assigned consistently."
    )

Robust reference assignment and scoring completed.
Candidate files scored: 10,038

Reference-assignment checks:
reference_groups_are_unique: True
all_files_receive_a_reference: True
reference_statistics_are_complete: True


In [28]:
# =============================================================================
# Characterize robust methylation-QC score distributions
# =============================================================================

modified_z_columns = [
    f"{metric}_modified_z"
    for metric in QC_REFERENCE_METRICS
]

score_checks = {
    "all_scores_are_complete": (
        methylation_file_qc_annotated[
            modified_z_columns
        ]
        .notna()
        .all()
        .all()
    ),
    "all_scores_are_finite": (
        np.isfinite(
            methylation_file_qc_annotated[
                modified_z_columns
            ].to_numpy(dtype=float)
        ).all()
    ),
}

print("Robust-score checks:")
for check_name, check_passed in score_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(score_checks.values()):
    raise ValueError(
        "One or more robust methylation-QC scores are invalid."
    )


# -----------------------------------------------------------------------------
# Absolute-score distributions
# -----------------------------------------------------------------------------

absolute_score_summary = (
    methylation_file_qc_annotated
    .assign(
        **{
            f"{column}_abs": (
                methylation_file_qc_annotated[column].abs()
            )
            for column in modified_z_columns
        }
    )
    .groupby(
        "methylation_platform",
        observed=True,
    )[
        [
            f"{column}_abs"
            for column in modified_z_columns
        ]
    ]
    .quantile(
        [0.50, 0.95, 0.99, 1.00]
    )
    .rename_axis(
        index=[
            "methylation_platform",
            "quantile",
        ]
    )
    .reset_index()
)

absolute_score_summary["quantile"] = (
    absolute_score_summary["quantile"]
    .map(
        lambda value: (
            "max"
            if value == 1
            else f"p{int(value * 100):02d}"
        )
    )
)

print("\nAbsolute modified-z distributions by platform:")
print(
    absolute_score_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)


# -----------------------------------------------------------------------------
# Exploratory tail counts
# -----------------------------------------------------------------------------

exploratory_thresholds = [
    3.5,
    5.0,
    7.5,
]

tail_count_records = []

for column in modified_z_columns:
    absolute_scores = (
        methylation_file_qc_annotated[column].abs()
    )

    for threshold in exploratory_thresholds:
        tail_count_records.append(
            {
                "metric": column,
                "absolute_threshold": threshold,
                "file_count": (
                    absolute_scores.ge(threshold).sum()
                ),
            }
        )

exploratory_tail_counts = pd.DataFrame(
    tail_count_records
)

print("\nExploratory absolute modified-z tail counts:")
print(
    exploratory_tail_counts.to_string(
        index=False,
    )
)

Robust-score checks:
all_scores_are_complete: True
all_scores_are_finite: True

Absolute modified-z distributions by platform:
          methylation_platform quantile  missing_beta_fraction_modified_z_abs  beta_median_modified_z_abs  beta_mean_modified_z_abs  beta_sd_modified_z_abs
 Illumina Human Methylation 27      p50                                 0.674                       0.674                     0.674                   0.674
 Illumina Human Methylation 27      p95                                 2.459                       2.855                     2.284                   2.145
 Illumina Human Methylation 27      p99                                 4.118                       4.911                     3.426                   3.069
 Illumina Human Methylation 27      max                                18.254                      20.110                     5.144                   5.049
Illumina Human Methylation 450      p50                                 0.674                

In [29]:
# =============================================================================
# Characterize direction-specific robust-score tails
# =============================================================================

directional_thresholds = [
    3.5,
    5.0,
    7.5,
]

directional_tail_records = []

for metric in QC_REFERENCE_METRICS:
    score_column = f"{metric}_modified_z"
    scores = methylation_file_qc_annotated[score_column]

    for threshold in directional_thresholds:
        directional_tail_records.extend(
            [
                {
                    "metric": metric,
                    "direction": "low",
                    "threshold": -threshold,
                    "file_count": scores.le(-threshold).sum(),
                },
                {
                    "metric": metric,
                    "direction": "high",
                    "threshold": threshold,
                    "file_count": scores.ge(threshold).sum(),
                },
            ]
        )

directional_tail_counts = pd.DataFrame(
    directional_tail_records
)

print("Direction-specific modified-z tail counts:")
print(
    directional_tail_counts.to_string(
        index=False,
    )
)

Direction-specific modified-z tail counts:


               metric direction  threshold  file_count
missing_beta_fraction       low       -3.5           0
missing_beta_fraction      high        3.5         566
missing_beta_fraction       low       -5.0           0
missing_beta_fraction      high        5.0         269
missing_beta_fraction       low       -7.5           0
missing_beta_fraction      high        7.5          91
          beta_median       low       -3.5          94
          beta_median      high        3.5          88
          beta_median       low       -5.0          22
          beta_median      high        5.0          21
          beta_median       low       -7.5           3
          beta_median      high        7.5           4
            beta_mean       low       -3.5         101
            beta_mean      high        3.5          42
            beta_mean       low       -5.0          30
            beta_mean      high        5.0           6
            beta_mean       low       -7.5           5
          

In [30]:
# =============================================================================
# Characterize severe exploratory methylation-QC signals
# =============================================================================

SEVERE_MISSINGNESS_Z_THRESHOLD = 7.5
LOW_BETA_SD_Z_THRESHOLD = -5.0
EXTREME_BETA_CENTER_Z_THRESHOLD = 7.5

methylation_file_qc_annotated[
    "severe_high_missingness"
] = (
    methylation_file_qc_annotated[
        "missing_beta_fraction_modified_z"
    ].ge(SEVERE_MISSINGNESS_Z_THRESHOLD)
)

methylation_file_qc_annotated[
    "severe_low_beta_sd"
] = (
    methylation_file_qc_annotated[
        "beta_sd_modified_z"
    ].le(LOW_BETA_SD_Z_THRESHOLD)
)

methylation_file_qc_annotated[
    "extreme_beta_center"
] = (
    methylation_file_qc_annotated[
        [
            "beta_median_modified_z",
            "beta_mean_modified_z",
        ]
    ]
    .abs()
    .ge(EXTREME_BETA_CENTER_Z_THRESHOLD)
    .any(axis=1)
)

exploratory_signal_columns = [
    "severe_high_missingness",
    "severe_low_beta_sd",
    "extreme_beta_center",
]

methylation_file_qc_annotated[
    "severe_qc_signal_count"
] = (
    methylation_file_qc_annotated[
        exploratory_signal_columns
    ]
    .sum(axis=1)
)

print("Severe exploratory QC signals:")
for column in exploratory_signal_columns:
    print(
        f"{column}: "
        f"{methylation_file_qc_annotated[column].sum():,}"
    )

print("\nNumber of severe signals per file:")
print(
    methylation_file_qc_annotated[
        "severe_qc_signal_count"
    ]
    .value_counts()
    .sort_index()
)

severe_signal_inventory = (
    methylation_file_qc_annotated.loc[
        methylation_file_qc_annotated[
            "severe_qc_signal_count"
        ].gt(0)
    ]
    .copy()
)

print("\nRaw missingness among severe-signal files:")
print(
    severe_signal_inventory
    .groupby(
        [
            "methylation_platform",
            "severe_high_missingness",
        ],
        observed=True,
    )["missing_beta_fraction"]
    .agg(
        file_count="size",
        minimum="min",
        median="median",
        maximum="max",
    )
    .reset_index()
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nProjects with the most severe-signal files:")
print(
    severe_signal_inventory[
        "project_id"
    ]
    .value_counts()
    .head(15)
)

Severe exploratory QC signals:
severe_high_missingness: 91
severe_low_beta_sd: 10
extreme_beta_center: 10

Number of severe signals per file:
severe_qc_signal_count
0    9929
1     107
2       2
Name: count, dtype: int64

Raw missingness among severe-signal files:
          methylation_platform  severe_high_missingness  file_count  minimum   median  maximum
 Illumina Human Methylation 27                    False           5 0.089782 0.096816 0.103017
 Illumina Human Methylation 27                     True           2 0.172202 0.236457 0.300711
Illumina Human Methylation 450                    False          13 0.136900 0.155810 0.193116
Illumina Human Methylation 450                     True          89 0.163704 0.203967 0.319921

Projects with the most severe-signal files:
project_id
TCGA-UCEC    20
TCGA-LUSC    14
TCGA-LUAD    13
TCGA-THCA    13
TCGA-HNSC    10
TCGA-KIRC     9
TCGA-PRAD     7
TCGA-COAD     5
TCGA-BRCA     3
TCGA-PAAD     3
TCGA-KIRP     2
TCGA-LGG      2
TCGA-SKCM   

In [31]:
# =============================================================================
# Characterize severe-signal prevalence by project and platform
# =============================================================================

project_platform_signal_summary = (
    methylation_file_qc_annotated
    .groupby(
        [
            "methylation_platform",
            "project_id",
        ],
        observed=True,
    )
    .agg(
        file_count=(
            "methylation_file_id",
            "size",
        ),
        high_missingness_count=(
            "severe_high_missingness",
            "sum",
        ),
        low_beta_sd_count=(
            "severe_low_beta_sd",
            "sum",
        ),
        extreme_beta_center_count=(
            "extreme_beta_center",
            "sum",
        ),
        any_severe_signal_count=(
            "severe_qc_signal_count",
            lambda values: values.gt(0).sum(),
        ),
    )
    .reset_index()
)

for count_column in [
    "high_missingness_count",
    "low_beta_sd_count",
    "extreme_beta_center_count",
    "any_severe_signal_count",
]:
    fraction_column = count_column.replace(
        "_count",
        "_fraction",
    )

    project_platform_signal_summary[
        fraction_column
    ] = (
        project_platform_signal_summary[count_column]
        / project_platform_signal_summary["file_count"]
    )

signal_prevalence_checks = {
    "file_counts_reconstruct_cohort": (
        project_platform_signal_summary[
            "file_count"
        ].sum()
        == len(methylation_file_qc_annotated)
    ),
    "signal_counts_do_not_exceed_stratum_size": (
        project_platform_signal_summary[
            "any_severe_signal_count"
        ]
        .le(
            project_platform_signal_summary[
                "file_count"
            ]
        )
        .all()
    ),
    "signal_fractions_are_valid": (
        project_platform_signal_summary[
            "any_severe_signal_fraction"
        ]
        .between(0, 1)
        .all()
    ),
}

print("Project-platform strata with severe QC signals:")
print(
    project_platform_signal_summary.loc[
        project_platform_signal_summary[
            "any_severe_signal_count"
        ].gt(0),
        [
            "methylation_platform",
            "project_id",
            "file_count",
            "high_missingness_count",
            "low_beta_sd_count",
            "extreme_beta_center_count",
            "any_severe_signal_count",
            "any_severe_signal_fraction",
        ],
    ]
    .sort_values(
        [
            "any_severe_signal_fraction",
            "any_severe_signal_count",
        ],
        ascending=False,
        kind="stable",
    )
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\nSignal-prevalence checks:")
for check_name, check_passed in signal_prevalence_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(signal_prevalence_checks.values()):
    raise ValueError(
        "Project-platform severe-signal prevalence is inconsistent."
    )

Project-platform strata with severe QC signals:
          methylation_platform project_id  file_count  high_missingness_count  low_beta_sd_count  extreme_beta_center_count  any_severe_signal_count  any_severe_signal_fraction
Illumina Human Methylation 450  TCGA-UCEC         437                      19                  0                          0                       19                      0.0435
Illumina Human Methylation 450  TCGA-LUSC         370                      10                  3                          1                       13                      0.0351
Illumina Human Methylation 450  TCGA-KIRC         323                       9                  0                          1                        9                      0.0279
Illumina Human Methylation 450  TCGA-THCA         505                      11                  1                          1                       13                      0.0257
 Illumina Human Methylation 27  TCGA-STAD          39              

In [32]:
# =============================================================================
# Assess comparator support for severe methylation-QC signals
# =============================================================================

comparator_group_keys = [
    "project_id",
    "case_submitter_id",
    "sample_submitter_id",
    "methylation_platform",
]

comparator_signal_inventory = (
    methylation_file_qc_annotated
    .merge(
        within_sample_comparator_groups[
            comparator_group_keys
        ],
        on=comparator_group_keys,
        how="inner",
        validate="many_to_one",
    )
)

comparator_signal_inventory[
    "group_min_missing_beta_fraction"
] = (
    comparator_signal_inventory
    .groupby(
        comparator_group_keys,
        observed=True,
    )["missing_beta_fraction"]
    .transform("min")
)

comparator_signal_inventory[
    "missingness_excess_over_best_comparator"
] = (
    comparator_signal_inventory[
        "missing_beta_fraction"
    ]
    - comparator_signal_inventory[
        "group_min_missing_beta_fraction"
    ]
)

comparator_signal_inventory[
    "group_severe_signal_count"
] = (
    comparator_signal_inventory
    .groupby(
        comparator_group_keys,
        observed=True,
    )["severe_qc_signal_count"]
    .transform(
        lambda values: values.gt(0).sum()
    )
)

severe_files_with_comparator = (
    comparator_signal_inventory.loc[
        comparator_signal_inventory[
            "severe_qc_signal_count"
        ].gt(0)
    ]
    .copy()
)

total_severe_files = (
    methylation_file_qc_annotated[
        "severe_qc_signal_count"
    ].gt(0).sum()
)

print("Comparator coverage of severe-signal files:")
print(f"Total severe-signal files:       {total_severe_files:,}")
print(
    "Severe files with comparator:   "
    f"{len(severe_files_with_comparator):,}"
)
print(
    "Severe files without comparator: "
    f"{total_severe_files - len(severe_files_with_comparator):,}"
)

if not severe_files_with_comparator.empty:
    print("\nSevere-signal files with within-sample comparators:")
    print(
        severe_files_with_comparator[
            [
                "project_id",
                "case_submitter_id",
                "methylation_file_id",
                "methylation_platform",
                "missing_beta_fraction",
                "missingness_excess_over_best_comparator",
                "severe_high_missingness",
                "severe_low_beta_sd",
                "extreme_beta_center",
                "group_severe_signal_count",
            ]
        ]
        .sort_values(
            "missingness_excess_over_best_comparator",
            ascending=False,
            kind="stable",
        )
        .to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}",
        )
    )

Comparator coverage of severe-signal files:
Total severe-signal files:       109
Severe files with comparator:   2
Severe files without comparator: 107

Severe-signal files with within-sample comparators:
project_id case_submitter_id                  methylation_file_id           methylation_platform  missing_beta_fraction  missingness_excess_over_best_comparator  severe_high_missingness  severe_low_beta_sd  extreme_beta_center  group_severe_signal_count
 TCGA-UCEC      TCGA-BK-A139 ee0b84d7-05e5-4490-a1c3-0c0b28a5b317 Illumina Human Methylation 450               0.202392                                 0.049407                     True               False                False                          1
 TCGA-UCEC      TCGA-BK-A0CA 310003b7-e63b-4238-b0e7-1432ca016c2d Illumina Human Methylation 450               0.224089                                 0.037319                     True               False                False                          1


In [33]:
# =============================================================================
# Compare severe-signal files with their within-sample comparators
# =============================================================================

severe_comparator_group_keys = (
    severe_files_with_comparator[
        comparator_group_keys
    ]
    .drop_duplicates()
)

pairwise_comparator_records = []

for group_record in severe_comparator_group_keys.itertuples(index=False):
    group_mask = (
        methylation_file_qc_annotated["project_id"]
        .eq(group_record.project_id)
        & methylation_file_qc_annotated["case_submitter_id"]
        .eq(group_record.case_submitter_id)
        & methylation_file_qc_annotated["sample_submitter_id"]
        .eq(group_record.sample_submitter_id)
        & methylation_file_qc_annotated["methylation_platform"]
        .eq(group_record.methylation_platform)
    )

    group_files = (
        methylation_file_qc_annotated.loc[group_mask]
        .sort_values(
            "missing_beta_fraction",
            ascending=False,
            kind="stable",
        )
        .reset_index(drop=True)
    )

    if len(group_files) != 2:
        raise ValueError(
            "Expected exactly two files in each severe comparator group."
        )

    higher_missingness_record = group_files.iloc[0]
    comparator_record = group_files.iloc[1]

    higher_missingness_payload = read_methylation_payload(
        methylation_candidate_inventory.loc[
            methylation_candidate_inventory[
                "methylation_file_id"
            ].eq(higher_missingness_record["methylation_file_id"]),
            "methylation_local_path",
        ].iloc[0]
    )

    comparator_payload = read_methylation_payload(
        methylation_candidate_inventory.loc[
            methylation_candidate_inventory[
                "methylation_file_id"
            ].eq(comparator_record["methylation_file_id"]),
            "methylation_local_path",
        ].iloc[0]
    )

    if not higher_missingness_payload["probe_id"].equals(
        comparator_payload["probe_id"]
    ):
        raise ValueError(
            "Probe identity or order differs within a comparator pair."
        )

    higher_beta = higher_missingness_payload["beta_value"]
    comparator_beta = comparator_payload["beta_value"]

    shared_numeric = (
        higher_beta.notna()
        & comparator_beta.notna()
    )

    absolute_beta_difference = (
        higher_beta.loc[shared_numeric]
        - comparator_beta.loc[shared_numeric]
    ).abs()

    pairwise_comparator_records.append(
        {
            "project_id": group_record.project_id,
            "case_submitter_id": group_record.case_submitter_id,
            "sample_submitter_id": group_record.sample_submitter_id,
            "higher_missingness_file_id": (
                higher_missingness_record["methylation_file_id"]
            ),
            "comparator_file_id": (
                comparator_record["methylation_file_id"]
            ),
            "higher_missingness_fraction": (
                higher_missingness_record["missing_beta_fraction"]
            ),
            "comparator_missingness_fraction": (
                comparator_record["missing_beta_fraction"]
            ),
            "missingness_fraction_delta": (
                higher_missingness_record["missing_beta_fraction"]
                - comparator_record["missing_beta_fraction"]
            ),
            "shared_numeric_probe_count": shared_numeric.sum(),
            "higher_file_only_missing_count": (
                higher_beta.isna()
                & comparator_beta.notna()
            ).sum(),
            "comparator_only_missing_count": (
                comparator_beta.isna()
                & higher_beta.notna()
            ).sum(),
            "pearson_beta_correlation": (
                higher_beta.loc[shared_numeric]
                .corr(comparator_beta.loc[shared_numeric])
            ),
            "median_absolute_beta_difference": (
                absolute_beta_difference.median()
            ),
            "p95_absolute_beta_difference": (
                absolute_beta_difference.quantile(0.95)
            ),
        }
    )

pairwise_severe_comparator_qc = pd.DataFrame(
    pairwise_comparator_records
)

print("Pairwise severe-signal comparator diagnostics:")
print(
    pairwise_severe_comparator_qc.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

Pairwise severe-signal comparator diagnostics:
project_id case_submitter_id sample_submitter_id           higher_missingness_file_id                   comparator_file_id  higher_missingness_fraction  comparator_missingness_fraction  missingness_fraction_delta  shared_numeric_probe_count  higher_file_only_missing_count  comparator_only_missing_count  pearson_beta_correlation  median_absolute_beta_difference  p95_absolute_beta_difference
 TCGA-UCEC      TCGA-BK-A0CA    TCGA-BK-A0CA-01A 310003b7-e63b-4238-b0e7-1432ca016c2d 1603a868-024c-40b9-bd8d-1249fe9fae67                     0.224089                         0.186770                    0.037319                      376259                           19318                           1165                  0.992786                         0.021358                      0.100629
 TCGA-UCEC      TCGA-BK-A139    TCGA-BK-A139-01A ee0b84d7-05e5-4490-a1c3-0c0b28a5b317 1339d46e-0015-4a5d-8120-17e90d0d0b94                     0.202392                

In [34]:
# =============================================================================
# Define methylation-QC eligibility policy
# =============================================================================

PAIRED_UNDERPERFORMANCE_MIN_MISSINGNESS_DELTA = 0.03
PAIRED_UNDERPERFORMANCE_MIN_CORRELATION = 0.99
PAIRED_UNDERPERFORMANCE_MIN_MISSINGNESS_ASYMMETRY = 10.0

pairwise_severe_comparator_qc[
    "missingness_asymmetry_ratio"
] = (
    pairwise_severe_comparator_qc[
        "higher_file_only_missing_count"
    ]
    / pairwise_severe_comparator_qc[
        "comparator_only_missing_count"
    ].replace(0, np.nan)
)

pairwise_severe_comparator_qc[
    "paired_technical_underperformance"
] = (
    pairwise_severe_comparator_qc[
        "missingness_fraction_delta"
    ].ge(PAIRED_UNDERPERFORMANCE_MIN_MISSINGNESS_DELTA)
    & pairwise_severe_comparator_qc[
        "pearson_beta_correlation"
    ].ge(PAIRED_UNDERPERFORMANCE_MIN_CORRELATION)
    & pairwise_severe_comparator_qc[
        "missingness_asymmetry_ratio"
    ].ge(PAIRED_UNDERPERFORMANCE_MIN_MISSINGNESS_ASYMMETRY)
)

paired_underperforming_file_ids = set(
    pairwise_severe_comparator_qc.loc[
        pairwise_severe_comparator_qc[
            "paired_technical_underperformance"
        ],
        "higher_missingness_file_id",
    ]
)

methylation_file_qc_annotated[
    "methylation_qc_eligible_for_downstream_selection"
] = (
    ~methylation_file_qc_annotated[
        "methylation_file_id"
    ].isin(paired_underperforming_file_ids)
)

methylation_file_qc_annotated[
    "methylation_qc_eligibility_status"
] = np.where(
    methylation_file_qc_annotated[
        "methylation_qc_eligible_for_downstream_selection"
    ],
    "Eligible after methylation QC",
    "Ineligible — paired technical underperformance",
)

methylation_file_qc_annotated[
    "methylation_qc_ineligibility_reason"
] = np.where(
    methylation_file_qc_annotated[
        "methylation_qc_eligible_for_downstream_selection"
    ],
    pd.NA,
    (
        "Severe platform-aware missingness outlier with a "
        "same-sample, same-platform comparator showing materially "
        "better probe coverage and highly concordant shared beta values"
    ),
)

methylation_file_qc_annotated[
    "methylation_qc_review_flag"
] = (
    methylation_file_qc_annotated[
        "severe_qc_signal_count"
    ].gt(0)
    & methylation_file_qc_annotated[
        "methylation_qc_eligible_for_downstream_selection"
    ]
)

policy_checks = {
    "paired_underperformance_files_are_unique": (
        len(paired_underperforming_file_ids)
        == pairwise_severe_comparator_qc[
            "paired_technical_underperformance"
        ].sum()
    ),
    "ineligible_files_have_reasons": (
        methylation_file_qc_annotated.loc[
            ~methylation_file_qc_annotated[
                "methylation_qc_eligible_for_downstream_selection"
            ],
            "methylation_qc_ineligibility_reason",
        ]
        .notna()
        .all()
    ),
    "eligible_files_have_no_ineligibility_reason": (
        methylation_file_qc_annotated.loc[
            methylation_file_qc_annotated[
                "methylation_qc_eligible_for_downstream_selection"
            ],
            "methylation_qc_ineligibility_reason",
        ]
        .isna()
        .all()
    ),
}

print("Methylation-QC eligibility policy applied.")
print(
    "Eligible files:   "
    f"{methylation_file_qc_annotated['methylation_qc_eligible_for_downstream_selection'].sum():,}"
)
print(
    "Ineligible files: "
    f"{(~methylation_file_qc_annotated['methylation_qc_eligible_for_downstream_selection']).sum():,}"
)
print(
    "Eligible files retaining review flags: "
    f"{methylation_file_qc_annotated['methylation_qc_review_flag'].sum():,}"
)

print("\nEligibility status:")
print(
    methylation_file_qc_annotated[
        "methylation_qc_eligibility_status"
    ].value_counts()
)

print("\nPolicy checks:")
for check_name, check_passed in policy_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(policy_checks.values()):
    raise ValueError(
        "Methylation-QC eligibility policy is inconsistent."
    )


Methylation-QC eligibility policy applied.
Eligible files:   10,036
Ineligible files: 2
Eligible files retaining review flags: 107

Eligibility status:
methylation_qc_eligibility_status
Eligible after methylation QC                     10036
Ineligible — paired technical underperformance        2
Name: count, dtype: int64

Policy checks:
paired_underperformance_files_are_unique: True
ineligible_files_have_reasons: True
eligible_files_have_no_ineligibility_reason: True


In [35]:
# =============================================================================
# Propagate methylation-QC status to candidate pairs
# =============================================================================

methylation_qc_columns_for_pairs = [
    "methylation_file_id",
    "missing_beta_fraction",
    "missing_beta_fraction_modified_z",
    "severe_high_missingness",
    "severe_low_beta_sd",
    "extreme_beta_center",
    "severe_qc_signal_count",
    "methylation_qc_review_flag",
    "methylation_qc_eligible_for_downstream_selection",
    "methylation_qc_eligibility_status",
    "methylation_qc_ineligibility_reason",
]

pair_inventory_with_methylation_qc = (
    rna_qc_pair_inventory
    .assign(_pair_row_order=np.arange(len(rna_qc_pair_inventory)))
    .merge(
        methylation_file_qc_annotated[
            methylation_qc_columns_for_pairs
        ],
        on="methylation_file_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    .sort_values(
        "_pair_row_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

pair_inventory_with_methylation_qc[
    "pair_eligible_after_methylation_qc"
] = (
    pair_inventory_with_methylation_qc[
        "pair_eligible_after_rna_qc"
    ]
    & pair_inventory_with_methylation_qc[
        "methylation_qc_eligible_for_downstream_selection"
    ]
)

pair_inventory_with_methylation_qc[
    "pair_methylation_qc_status"
] = np.select(
    [
        ~pair_inventory_with_methylation_qc[
            "pair_eligible_after_rna_qc"
        ],
        ~pair_inventory_with_methylation_qc[
            "methylation_qc_eligible_for_downstream_selection"
        ],
    ],
    [
        "Ineligible candidate pair — RNA-seq QC",
        "Ineligible candidate pair — methylation QC",
    ],
    default=(
        "Eligible candidate pair after RNA-seq "
        "and methylation QC"
    ),
)

propagation_checks = {
    "row_count_is_preserved": (
        len(pair_inventory_with_methylation_qc)
        == len(rna_qc_pair_inventory)
    ),
    "pair_order_is_preserved": (
        pair_inventory_with_methylation_qc[
            "_pair_row_order"
        ].eq(np.arange(len(rna_qc_pair_inventory))).all()
    ),
    "all_pairs_match_file_qc": (
        pair_inventory_with_methylation_qc["_merge"]
        .eq("both")
        .all()
    ),
    "methylation_eligibility_is_complete": (
        pair_inventory_with_methylation_qc[
            "methylation_qc_eligible_for_downstream_selection"
        ].notna().all()
    ),
    "final_pair_gate_is_boolean": (
        pd.api.types.is_bool_dtype(
            pair_inventory_with_methylation_qc[
                "pair_eligible_after_methylation_qc"
            ]
        )
    ),
}

pair_inventory_with_methylation_qc = (
    pair_inventory_with_methylation_qc
    .drop(
        columns=[
            "_pair_row_order",
            "_merge",
        ]
    )
)

print("Methylation-QC status propagated to candidate pairs.")
print(
    "\nFinal pair eligibility:"
)
print(
    pair_inventory_with_methylation_qc[
        "pair_eligible_after_methylation_qc"
    ].value_counts(dropna=False)
)

print("\nPair status:")
print(
    pair_inventory_with_methylation_qc[
        "pair_methylation_qc_status"
    ].value_counts(dropna=False)
)

eligible_review_flag_count = (
    pair_inventory_with_methylation_qc[
        "pair_eligible_after_methylation_qc"
    ]
    & pair_inventory_with_methylation_qc[
        "methylation_qc_review_flag"
    ]
).sum()

print(
    "\nEligible pairs retaining methylation review flags: "
    f"{eligible_review_flag_count:,}"
)

print("\nPropagation checks:")
for check_name, check_passed in propagation_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(propagation_checks.values()):
    raise ValueError(
        "Methylation-QC propagation to candidate pairs "
        "is inconsistent."
    )

Methylation-QC status propagated to candidate pairs.

Final pair eligibility:
pair_eligible_after_methylation_qc
True     10154
False        8
Name: count, dtype: int64

Pair status:
pair_methylation_qc_status
Eligible candidate pair after RNA-seq and methylation QC    10154
Ineligible candidate pair — RNA-seq QC                          4
Ineligible candidate pair — methylation QC                      4
Name: count, dtype: int64

Eligible pairs retaining methylation review flags: 108

Propagation checks:
row_count_is_preserved: True
pair_order_is_preserved: True
all_pairs_match_file_qc: True
methylation_eligibility_is_complete: True
final_pair_gate_is_boolean: True


In [36]:
# =============================================================================
# Audit final candidate-pair exclusions
# =============================================================================

rna_qc_ineligible_mask = (
    ~pair_inventory_with_methylation_qc[
        "pair_eligible_after_rna_qc"
    ]
)

methylation_qc_ineligible_mask = (
    pair_inventory_with_methylation_qc[
        "pair_eligible_after_rna_qc"
    ]
    & ~pair_inventory_with_methylation_qc[
        "methylation_qc_eligible_for_downstream_selection"
    ]
)

final_ineligible_mask = (
    ~pair_inventory_with_methylation_qc[
        "pair_eligible_after_methylation_qc"
    ]
)


# -----------------------------------------------------------------------------
# Compute exclusion counts
# -----------------------------------------------------------------------------

rna_qc_ineligible_pair_count = int(
    rna_qc_ineligible_mask.sum()
)

methylation_qc_ineligible_pair_count = int(
    methylation_qc_ineligible_mask.sum()
)

final_ineligible_pair_count = int(
    final_ineligible_mask.sum()
)

excluded_methylation_file_count = (
    pair_inventory_with_methylation_qc.loc[
        methylation_qc_ineligible_mask,
        "methylation_file_id",
    ]
    .nunique()
)

methylation_qc_affected_case_count = (
    pair_inventory_with_methylation_qc.loc[
        methylation_qc_ineligible_mask,
        "case_submitter_id",
    ]
    .nunique()
)


# -----------------------------------------------------------------------------
# Validate exclusion reconstruction
# -----------------------------------------------------------------------------

exclusion_audit_checks = {
    "final_ineligible_count_reconstructs": (
        final_ineligible_pair_count
        == (
            rna_qc_ineligible_pair_count
            + methylation_qc_ineligible_pair_count
        )
    ),
    "rna_and_methylation_exclusions_do_not_overlap": (
        ~(
            rna_qc_ineligible_mask
            & methylation_qc_ineligible_mask
        ).any()
    ),
    "all_final_ineligible_pairs_have_a_status": (
        pair_inventory_with_methylation_qc.loc[
            final_ineligible_mask,
            "pair_methylation_qc_status",
        ]
        .notna()
        .all()
    ),
}

print("Final candidate-pair exclusion audit:")
print(
    f"RNA-seq-QC-ineligible pairs:      "
    f"{rna_qc_ineligible_pair_count:,}"
)
print(
    f"Methylation-QC-ineligible pairs:  "
    f"{methylation_qc_ineligible_pair_count:,}"
)
print(
    f"Total ineligible pairs:           "
    f"{final_ineligible_pair_count:,}"
)
print(
    f"\nUnique methylation files excluded: "
    f"{excluded_methylation_file_count:,}"
)
print(
    f"Cases affected by methylation QC:  "
    f"{methylation_qc_affected_case_count:,}"
)

print("\nExclusion-audit checks:")
for check_name, check_passed in exclusion_audit_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(exclusion_audit_checks.values()):
    raise ValueError(
        "Final candidate-pair exclusions are inconsistent."
    )

Final candidate-pair exclusion audit:
RNA-seq-QC-ineligible pairs:      4
Methylation-QC-ineligible pairs:  4
Total ineligible pairs:           8

Unique methylation files excluded: 2
Cases affected by methylation QC:  2

Exclusion-audit checks:
final_ineligible_count_reconstructs: True
rna_and_methylation_exclusions_do_not_overlap: True
all_final_ineligible_pairs_have_a_status: True


In [37]:
# =============================================================================
# Save methylation-QC-annotated candidate-pair inventory
# =============================================================================

METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH = (
    METHYLATION_METADATA_OUTPUT_DIR
    / (
        "tcga_primary_tumor_rnaseq_methylation_"
        "qc_annotated_candidate_pair_inventory.csv"
    )
)

pair_identity_columns = [
    "rna_file_id",
    "methylation_file_id",
]

required_output_columns = [
    *pair_identity_columns,
    "pair_eligible_after_rna_qc",
    "methylation_qc_eligible_for_downstream_selection",
    "methylation_qc_review_flag",
    "pair_eligible_after_methylation_qc",
    "pair_methylation_qc_status",
]

missing_output_columns = [
    column
    for column in required_output_columns
    if column not in pair_inventory_with_methylation_qc.columns
]

pre_write_checks = {
    "inventory_is_not_empty": (
        not pair_inventory_with_methylation_qc.empty
    ),
    "required_columns_are_present": (
        not missing_output_columns
    ),
    "candidate_file_pairs_are_unique": (
        not pair_inventory_with_methylation_qc[
            pair_identity_columns
        ].duplicated().any()
    ),
    "final_pair_gate_is_complete": (
        pair_inventory_with_methylation_qc[
            "pair_eligible_after_methylation_qc"
        ].notna().all()
    ),
    "final_pair_gate_is_boolean": (
        pd.api.types.is_bool_dtype(
            pair_inventory_with_methylation_qc[
                "pair_eligible_after_methylation_qc"
            ]
        )
    ),
    "pair_counts_reconstruct_total": (
        pair_inventory_with_methylation_qc[
            "pair_eligible_after_methylation_qc"
        ].value_counts(dropna=False).sum()
        == len(pair_inventory_with_methylation_qc)
    ),
}

print("Candidate-pair inventory pre-write checks:")
for check_name, check_passed in pre_write_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(pre_write_checks.values()):
    raise ValueError(
        "Methylation-QC-annotated candidate-pair inventory "
        "failed pre-write validation."
    )


# -----------------------------------------------------------------------------
# Atomic write and round-trip validation
# -----------------------------------------------------------------------------

METHYLATION_METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

temporary_output_path = (
    METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH.with_name(
        METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH.name
        + ".tmp"
    )
)

pair_inventory_with_methylation_qc.to_csv(
    temporary_output_path,
    index=False,
)

written_pair_inventory = pd.read_csv(
    temporary_output_path,
    low_memory=False,
)

written_checks = {
    "temporary_output_exists": (
        temporary_output_path.is_file()
    ),
    "written_shape_matches_memory": (
        written_pair_inventory.shape
        == pair_inventory_with_methylation_qc.shape
    ),
    "written_columns_match_memory": (
        written_pair_inventory.columns.tolist()
        == pair_inventory_with_methylation_qc.columns.tolist()
    ),
    "written_pair_identity_and_order_match_memory": (
        written_pair_inventory[
            pair_identity_columns
        ].astype("string").equals(
            pair_inventory_with_methylation_qc[
                pair_identity_columns
            ].astype("string")
        )
    ),
    "written_final_gate_matches_memory": (
        written_pair_inventory[
            "pair_eligible_after_methylation_qc"
        ].astype("boolean").equals(
            pair_inventory_with_methylation_qc[
                "pair_eligible_after_methylation_qc"
            ].astype("boolean")
        )
    ),
    "written_pair_status_matches_memory": (
        written_pair_inventory[
            "pair_methylation_qc_status"
        ].astype("string").equals(
            pair_inventory_with_methylation_qc[
                "pair_methylation_qc_status"
            ].astype("string")
        )
    ),
}

print("\nWritten-artifact checks:")
for check_name, check_passed in written_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(written_checks.values()):
    temporary_output_path.unlink(missing_ok=True)

    raise IOError(
        "Written candidate-pair inventory failed "
        "round-trip validation."
    )

temporary_output_path.replace(
    METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
)

eligible_pair_count = int(
    pair_inventory_with_methylation_qc[
        "pair_eligible_after_methylation_qc"
    ].sum()
)

ineligible_pair_count = (
    len(pair_inventory_with_methylation_qc)
    - eligible_pair_count
)

output_relative_path = project_relative_path(
    METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
)

output_shape = pair_inventory_with_methylation_qc.shape

output_file_size_mib = (
    METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
    .stat()
    .st_size
    / (1024**2)
)

print(
    "\nMethylation-QC-annotated candidate-pair "
    "inventory saved."
)
print(f"Path: {output_relative_path}")
print(f"Shape: {output_shape}")
print(f"Eligible pairs:   {eligible_pair_count:,}")
print(f"Ineligible pairs: {ineligible_pair_count:,}")
print(f"File size: {output_file_size_mib:,.3f} MiB")

Candidate-pair inventory pre-write checks:
inventory_is_not_empty: True
required_columns_are_present: True
candidate_file_pairs_are_unique: True
final_pair_gate_is_complete: True
final_pair_gate_is_boolean: True
pair_counts_reconstruct_total: True

Written-artifact checks:
temporary_output_exists: True
written_shape_matches_memory: True
written_columns_match_memory: True
written_pair_identity_and_order_match_memory: True
written_final_gate_matches_memory: True
written_pair_status_matches_memory: True

Methylation-QC-annotated candidate-pair inventory saved.
Path: data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_qc_annotated_candidate_pair_inventory.csv
Shape: (10162, 63)
Eligible pairs:   10,154
Ineligible pairs: 8
File size: 9.431 MiB


In [38]:
# =============================================================================
# Summarize downstream cohort impact after methylation QC
# =============================================================================

rna_qc_eligible_pairs = (
    pair_inventory_with_methylation_qc.loc[
        pair_inventory_with_methylation_qc[
            "pair_eligible_after_rna_qc"
        ]
    ]
)

final_eligible_pairs = (
    pair_inventory_with_methylation_qc.loc[
        pair_inventory_with_methylation_qc[
            "pair_eligible_after_methylation_qc"
        ]
    ]
)

methylation_qc_excluded_pairs = (
    pair_inventory_with_methylation_qc.loc[
        pair_inventory_with_methylation_qc[
            "pair_methylation_qc_status"
        ].eq("Ineligible candidate pair — methylation QC")
    ]
)


# -----------------------------------------------------------------------------
# Cohort counts
# -----------------------------------------------------------------------------

pre_qc_case_ids = set(
    rna_qc_eligible_pairs["case_submitter_id"]
)

post_qc_case_ids = set(
    final_eligible_pairs["case_submitter_id"]
)

excluded_methylation_file_ids = set(
    methylation_qc_excluded_pairs[
        "methylation_file_id"
    ]
)

eligible_methylation_file_ids = set(
    final_eligible_pairs[
        "methylation_file_id"
    ]
)

case_loss_count = len(
    pre_qc_case_ids - post_qc_case_ids
)

affected_case_ids = set(
    methylation_qc_excluded_pairs[
        "case_submitter_id"
    ]
)

affected_cases_with_eligible_alternative = (
    affected_case_ids.issubset(post_qc_case_ids)
)


# -----------------------------------------------------------------------------
# Validate downstream impact
# -----------------------------------------------------------------------------

cohort_impact_checks = {
    "final_eligible_pair_count_matches_gate": (
        len(final_eligible_pairs)
        == pair_inventory_with_methylation_qc[
            "pair_eligible_after_methylation_qc"
        ].sum()
    ),
    "excluded_methylation_files_are_absent_from_eligible_pairs": (
        excluded_methylation_file_ids.isdisjoint(
            eligible_methylation_file_ids
        )
    ),
    "affected_cases_retain_an_eligible_alternative": (
        affected_cases_with_eligible_alternative
    ),
    "methylation_qc_causes_no_case_loss": (
        case_loss_count == 0
    ),
}

print("Downstream cohort impact after methylation QC:")
print(
    f"Eligible candidate pairs:      "
    f"{len(final_eligible_pairs):,}"
)
print(
    f"Eligible RNA-seq files:        "
    f"{final_eligible_pairs['rna_file_id'].nunique():,}"
)
print(
    f"Eligible methylation files:    "
    f"{final_eligible_pairs['methylation_file_id'].nunique():,}"
)
print(
    f"Cases retained:                "
    f"{final_eligible_pairs['case_submitter_id'].nunique():,}"
)
print(
    f"Projects retained:             "
    f"{final_eligible_pairs['project_id'].nunique():,}"
)
print(
    f"Cases lost through methylation QC: "
    f"{case_loss_count:,}"
)

print("\nEligible methylation files by platform:")
print(
    final_eligible_pairs[
        [
            "methylation_file_id",
            "methylation_platform",
        ]
    ]
    .drop_duplicates()
    ["methylation_platform"]
    .value_counts()
)

print("\nCohort-impact checks:")
for check_name, check_passed in cohort_impact_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(cohort_impact_checks.values()):
    raise ValueError(
        "Unexpected cohort loss or eligibility inconsistency "
        "after methylation QC."
    )

Downstream cohort impact after methylation QC:
Eligible candidate pairs:      10,154
Eligible RNA-seq files:        10,119
Eligible methylation files:    10,036
Cases retained:                9,973
Projects retained:             33
Cases lost through methylation QC: 0

Eligible methylation files by platform:
methylation_platform
Illumina Human Methylation 450    8415
Illumina Human Methylation 27     1621
Name: count, dtype: int64

Cohort-impact checks:
final_eligible_pair_count_matches_gate: True
excluded_methylation_files_are_absent_from_eligible_pairs: True
affected_cases_retain_an_eligible_alternative: True
methylation_qc_causes_no_case_loss: True


In [39]:
# =============================================================================
# Characterize platform-specific and shared probe spaces
# =============================================================================

HM27_PLATFORM = "Illumina Human Methylation 27"
HM450_PLATFORM = "Illumina Human Methylation 450"

hm27_probe_ids = platform_reference_probe_ids[
    HM27_PLATFORM
]

hm450_probe_ids = platform_reference_probe_ids[
    HM450_PLATFORM
]

shared_probe_ids = hm27_probe_ids.intersection(
    hm450_probe_ids,
    sort=False,
)

hm27_only_probe_ids = hm27_probe_ids.difference(
    hm450_probe_ids,
    sort=False,
)

hm450_only_probe_ids = hm450_probe_ids.difference(
    hm27_probe_ids,
    sort=False,
)

hm27_shared_order = hm27_probe_ids[
    hm27_probe_ids.isin(shared_probe_ids)
]

hm450_shared_order = hm450_probe_ids[
    hm450_probe_ids.isin(shared_probe_ids)
]

probe_space_summary = pd.DataFrame(
    [
        {
            "probe_space": "HM27 total",
            "probe_count": len(hm27_probe_ids),
        },
        {
            "probe_space": "HM450 total",
            "probe_count": len(hm450_probe_ids),
        },
        {
            "probe_space": "Shared HM27–HM450",
            "probe_count": len(shared_probe_ids),
        },
        {
            "probe_space": "HM27 only",
            "probe_count": len(hm27_only_probe_ids),
        },
        {
            "probe_space": "HM450 only",
            "probe_count": len(hm450_only_probe_ids),
        },
    ]
)

probe_space_checks = {
    "hm27_partition_reconstructs_total": (
        len(shared_probe_ids)
        + len(hm27_only_probe_ids)
        == len(hm27_probe_ids)
    ),
    "hm450_partition_reconstructs_total": (
        len(shared_probe_ids)
        + len(hm450_only_probe_ids)
        == len(hm450_probe_ids)
    ),
    "shared_probe_ids_are_unique": (
        shared_probe_ids.is_unique
    ),
    "shared_probe_order_matches_across_platforms": (
        hm27_shared_order.equals(
            hm450_shared_order
        )
    ),
}

print("Platform-specific probe-space summary:")
print(
    probe_space_summary.to_string(
        index=False,
    )
)

print("\nShared-space coverage:")
print(
    "HM27 probes represented in HM450: "
    f"{len(shared_probe_ids) / len(hm27_probe_ids):.4%}"
)
print(
    "HM450 probes represented in HM27: "
    f"{len(shared_probe_ids) / len(hm450_probe_ids):.4%}"
)

print("\nProbe-space checks:")
for check_name, check_passed in probe_space_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(
    check_passed
    for check_name, check_passed in probe_space_checks.items()
    if check_name != "shared_probe_order_matches_across_platforms"
):
    raise ValueError(
        "Platform-specific probe-space partitions are inconsistent."
    )

Platform-specific probe-space summary:
      probe_space  probe_count
       HM27 total        27578
      HM450 total       486427
Shared HM27–HM450        25978
        HM27 only         1600
       HM450 only       460449

Shared-space coverage:
HM27 probes represented in HM450: 94.1983%
HM450 probes represented in HM27: 5.3406%

Probe-space checks:
hm27_partition_reconstructs_total: True
hm450_partition_reconstructs_total: True
shared_probe_ids_are_unique: True
shared_probe_order_matches_across_platforms: True


In [40]:
# =============================================================================
# Construct the cross-platform probe-space inventory
# =============================================================================

all_probe_ids = pd.Index(
    pd.concat(
        [
            pd.Series(hm27_probe_ids),
            pd.Series(hm450_probe_ids),
        ],
        ignore_index=True,
    )
    .drop_duplicates(),
    name="probe_id",
)

probe_space_inventory = pd.DataFrame(
    {
        "probe_id": all_probe_ids,
    }
)

probe_space_inventory["present_in_hm27"] = (
    probe_space_inventory["probe_id"]
    .isin(hm27_probe_ids)
)

probe_space_inventory["present_in_hm450"] = (
    probe_space_inventory["probe_id"]
    .isin(hm450_probe_ids)
)

probe_space_inventory["probe_space_status"] = np.select(
    [
        (
            probe_space_inventory["present_in_hm27"]
            & probe_space_inventory["present_in_hm450"]
        ),
        (
            probe_space_inventory["present_in_hm27"]
            & ~probe_space_inventory["present_in_hm450"]
        ),
        (
            ~probe_space_inventory["present_in_hm27"]
            & probe_space_inventory["present_in_hm450"]
        ),
    ],
    [
        "shared_hm27_hm450",
        "hm27_only",
        "hm450_only",
    ],
    default="unresolved",
)

expected_union_size = (
    len(hm27_probe_ids)
    + len(hm450_probe_ids)
    - len(shared_probe_ids)
)

probe_inventory_checks = {
    "probe_ids_are_complete": (
        probe_space_inventory["probe_id"]
        .notna()
        .all()
    ),
    "probe_ids_are_unique": (
        probe_space_inventory["probe_id"]
        .is_unique
    ),
    "union_size_is_correct": (
        len(probe_space_inventory)
        == expected_union_size
    ),
    "all_probe_memberships_are_resolved": (
        probe_space_inventory["probe_space_status"]
        .ne("unresolved")
        .all()
    ),
}

print("Cross-platform probe-space inventory constructed.")
print(f"Union probe count: {len(probe_space_inventory):,}")

print("\nProbe-space membership:")
print(
    probe_space_inventory[
        "probe_space_status"
    ].value_counts()
)

print("\nProbe-inventory checks:")
for check_name, check_passed in probe_inventory_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(probe_inventory_checks.values()):
    raise ValueError(
        "Cross-platform probe-space inventory is inconsistent."
    )

Cross-platform probe-space inventory constructed.
Union probe count: 488,027

Probe-space membership:
probe_space_status
hm450_only           460449
shared_hm27_hm450     25978
hm27_only              1600
Name: count, dtype: int64

Probe-inventory checks:
probe_ids_are_complete: True
probe_ids_are_unique: True
union_size_is_correct: True
all_probe_memberships_are_resolved: True


In [41]:
# =============================================================================
# Prepare the eligible file inventory for probe-level QC
# =============================================================================

eligible_methylation_file_ids = set(
    methylation_file_qc_annotated.loc[
        methylation_file_qc_annotated[
            "methylation_qc_eligible_for_downstream_selection"
        ],
        "methylation_file_id",
    ]
)

eligible_methylation_file_inventory = (
    methylation_candidate_inventory.loc[
        methylation_candidate_inventory[
            "methylation_file_id"
        ].isin(eligible_methylation_file_ids)
    ]
    .sort_values(
        [
            "methylation_platform",
            "methylation_file_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

probe_qc_input_checks = {
    "eligible_file_ids_are_unique": (
        eligible_methylation_file_inventory[
            "methylation_file_id"
        ].is_unique
    ),
    "eligible_file_count_matches_file_qc": (
        len(eligible_methylation_file_inventory)
        == methylation_file_qc_annotated[
            "methylation_qc_eligible_for_downstream_selection"
        ].sum()
    ),
    "excluded_files_are_absent": (
        set(
            methylation_candidate_inventory[
                "methylation_file_id"
            ]
        )
        .difference(eligible_methylation_file_ids)
        .isdisjoint(
            set(
                eligible_methylation_file_inventory[
                    "methylation_file_id"
                ]
            )
        )
    ),
    "all_local_payloads_are_available": (
        eligible_methylation_file_inventory[
            "methylation_local_path"
        ]
        .map(lambda path: path.is_file())
        .all()
    ),
}

print("Eligible methylation files prepared for probe-level QC.")
print(
    f"Eligible files: "
    f"{len(eligible_methylation_file_inventory):,}"
)

print("\nEligible files by platform:")
print(
    eligible_methylation_file_inventory[
        "methylation_platform"
    ].value_counts()
)

print("\nProbe-level QC input checks:")
for check_name, check_passed in probe_qc_input_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(probe_qc_input_checks.values()):
    raise ValueError(
        "Eligible methylation-file inventory is inconsistent."
    )

Eligible methylation files prepared for probe-level QC.
Eligible files: 10,036

Eligible files by platform:
methylation_platform
Illumina Human Methylation 450    8415
Illumina Human Methylation 27     1621
Name: count, dtype: int64

Probe-level QC input checks:
eligible_file_ids_are_unique: True
eligible_file_count_matches_file_qc: True
excluded_files_are_absent: True
all_local_payloads_are_available: True


In [42]:
# =============================================================================
# Configure resumable probe-level methylation QC
# =============================================================================

PROBE_QC_CHECKPOINT_DIR = (
    METHYLATION_QC_OUTPUT_DIR
    / "tcga_primary_tumor_methylation_probe_qc_checkpoints"
)

PROBE_QC_CHECKPOINT_INTERVAL = 100

PLATFORM_SLUGS = {
    HM27_PLATFORM: "hm27",
    HM450_PLATFORM: "hm450",
}

probe_qc_checkpoint_paths = {
    platform: (
        PROBE_QC_CHECKPOINT_DIR
        / f"{platform_slug}_probe_missingness_checkpoint.npz"
    )
    for platform, platform_slug in PLATFORM_SLUGS.items()
}

probe_qc_platform_inventories = {
    platform: (
        eligible_methylation_file_inventory.loc[
            eligible_methylation_file_inventory[
                "methylation_platform"
            ].eq(platform)
        ]
        .sort_values(
            "methylation_file_id",
            kind="stable",
        )
        .reset_index(drop=True)
    )
    for platform in PLATFORM_SLUGS
}

PROBE_QC_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

configuration_checks = {
    "both_platforms_are_configured": (
        set(probe_qc_platform_inventories)
        == set(platform_reference_probe_ids)
    ),
    "platform_inventories_are_not_empty": all(
        not inventory.empty
        for inventory in probe_qc_platform_inventories.values()
    ),
    "platform_file_ids_are_unique": all(
        inventory["methylation_file_id"].is_unique
        for inventory in probe_qc_platform_inventories.values()
    ),
    "platform_counts_reconstruct_eligible_inventory": (
        sum(
            len(inventory)
            for inventory in probe_qc_platform_inventories.values()
        )
        == len(eligible_methylation_file_inventory)
    ),
}

print("Resumable probe-level QC configured.")
print(
    "Checkpoint directory: "
    f"{project_relative_path(PROBE_QC_CHECKPOINT_DIR)}"
)
print(
    f"Checkpoint interval: "
    f"{PROBE_QC_CHECKPOINT_INTERVAL:,} files"
)

print("\nPlatform workloads:")
for platform, inventory in probe_qc_platform_inventories.items():
    print(
        f"{platform}: "
        f"{len(inventory):,} files, "
        f"{len(platform_reference_probe_ids[platform]):,} probes"
    )
    print(
        "  Checkpoint: "
        f"{project_relative_path(probe_qc_checkpoint_paths[platform])}"
    )

print("\nConfiguration checks:")
for check_name, check_passed in configuration_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(configuration_checks.values()):
    raise ValueError(
        "Probe-level QC checkpoint configuration is inconsistent."
    )

Resumable probe-level QC configured.
Checkpoint directory: data/interim/qc/tcga_primary_tumor_methylation_probe_qc_checkpoints
Checkpoint interval: 100 files

Platform workloads:
Illumina Human Methylation 27: 1,621 files, 27,578 probes
  Checkpoint: data/interim/qc/tcga_primary_tumor_methylation_probe_qc_checkpoints/hm27_probe_missingness_checkpoint.npz
Illumina Human Methylation 450: 8,415 files, 486,427 probes
  Checkpoint: data/interim/qc/tcga_primary_tumor_methylation_probe_qc_checkpoints/hm450_probe_missingness_checkpoint.npz

Configuration checks:
both_platforms_are_configured: True
platform_inventories_are_not_empty: True
platform_file_ids_are_unique: True
platform_counts_reconstruct_eligible_inventory: True


In [43]:
# =============================================================================
# Define probe-level QC checkpoint utilities
# =============================================================================

PROBE_QC_CHECKPOINT_SCHEMA_VERSION = "1.0"


def initialize_probe_qc_checkpoint(platform):
    probe_count = len(
        platform_reference_probe_ids[platform]
    )

    return {
        "processed_file_ids": [],
        "missing_beta_counts": np.zeros(
            probe_count,
            dtype=np.uint32,
        ),
    }


def save_probe_qc_checkpoint(
    platform,
    processed_file_ids,
    missing_beta_counts,
):
    checkpoint_path = probe_qc_checkpoint_paths[
        platform
    ]

    temporary_path = checkpoint_path.with_name(
        checkpoint_path.stem + ".tmp.npz"
    )

    np.savez_compressed(
        temporary_path,
        schema_version=np.array(
            PROBE_QC_CHECKPOINT_SCHEMA_VERSION
        ),
        platform=np.array(platform),
        probe_count=np.array(
            len(platform_reference_probe_ids[platform]),
            dtype=np.int64,
        ),
        processed_file_ids=np.asarray(
            processed_file_ids,
            dtype=str,
        ),
        missing_beta_counts=np.asarray(
            missing_beta_counts,
            dtype=np.uint32,
        ),
    )

    temporary_path.replace(checkpoint_path)


def load_probe_qc_checkpoint(platform):
    checkpoint_path = probe_qc_checkpoint_paths[
        platform
    ]

    if not checkpoint_path.is_file():
        return initialize_probe_qc_checkpoint(
            platform
        )

    with np.load(
        checkpoint_path,
        allow_pickle=False,
    ) as checkpoint:
        schema_version = str(
            checkpoint["schema_version"].item()
        )
        stored_platform = str(
            checkpoint["platform"].item()
        )
        stored_probe_count = int(
            checkpoint["probe_count"].item()
        )

        processed_file_ids = (
            checkpoint["processed_file_ids"]
            .astype(str)
            .tolist()
        )

        missing_beta_counts = (
            checkpoint["missing_beta_counts"]
            .astype(np.uint32)
        )

    expected_probe_count = len(
        platform_reference_probe_ids[platform]
    )

    checkpoint_checks = {
        "schema_version_matches": (
            schema_version
            == PROBE_QC_CHECKPOINT_SCHEMA_VERSION
        ),
        "platform_matches": (
            stored_platform == platform
        ),
        "probe_count_matches": (
            stored_probe_count == expected_probe_count
        ),
        "missing_count_vector_matches_probe_count": (
            len(missing_beta_counts)
            == expected_probe_count
        ),
    }

    if not all(checkpoint_checks.values()):
        failed_checks = [
            check_name
            for check_name, check_passed
            in checkpoint_checks.items()
            if not check_passed
        ]

        raise ValueError(
            "Invalid probe-QC checkpoint for "
            f"{platform}: "
            + ", ".join(failed_checks)
        )

    return {
        "processed_file_ids": processed_file_ids,
        "missing_beta_counts": missing_beta_counts,
    }

In [44]:
# =============================================================================
# Define the resumable probe-level QC processor
# =============================================================================

def process_probe_qc_platform(platform):
    inventory = probe_qc_platform_inventories[
        platform
    ]

    reference_probe_ids = (
        platform_reference_probe_ids[
            platform
        ]
    )

    inventory_file_ids = (
        inventory["methylation_file_id"]
        .astype(str)
        .tolist()
    )

    checkpoint = load_probe_qc_checkpoint(
        platform
    )

    processed_file_ids = checkpoint[
        "processed_file_ids"
    ]

    missing_beta_counts = checkpoint[
        "missing_beta_counts"
    ]

    processed_file_count = len(
        processed_file_ids
    )

    total_file_count = len(
        inventory_file_ids
    )


    # -------------------------------------------------------------------------
    # Validate checkpoint compatibility
    # -------------------------------------------------------------------------

    expected_processed_prefix = (
        inventory_file_ids[
            :processed_file_count
        ]
    )

    checkpoint_state_checks = {
        "processed_count_does_not_exceed_inventory": (
            processed_file_count
            <= total_file_count
        ),
        "processed_file_ids_are_unique": (
            len(processed_file_ids)
            == len(set(processed_file_ids))
        ),
        "processed_file_ids_match_inventory_prefix": (
            processed_file_ids
            == expected_processed_prefix
        ),
        "missing_counts_are_integer": (
            np.issubdtype(
                missing_beta_counts.dtype,
                np.integer,
            )
        ),
        "missing_counts_do_not_exceed_processed_files": (
            missing_beta_counts.max(
                initial=0
            )
            <= processed_file_count
        ),
    }

    if not all(checkpoint_state_checks.values()):
        failed_checks = [
            check_name
            for check_name, check_passed
            in checkpoint_state_checks.items()
            if not check_passed
        ]

        raise ValueError(
            f"Invalid checkpoint state for {platform}: "
            + ", ".join(failed_checks)
        )

    print(f"Platform: {platform}")
    print(
        f"Checkpoint progress: "
        f"{processed_file_count:,}/{total_file_count:,}"
    )

    if processed_file_count == total_file_count:
        print("Probe-level QC is already complete.")

        return {
            "platform": platform,
            "processed_file_ids": processed_file_ids,
            "missing_beta_counts": missing_beta_counts,
        }


    # -------------------------------------------------------------------------
    # Process remaining payloads
    # -------------------------------------------------------------------------

    session_start_time = pd.Timestamp.now(
        tz="UTC"
    )

    for inventory_position in range(
        processed_file_count,
        total_file_count,
    ):
        record = inventory.iloc[
            inventory_position
        ]

        file_id = str(
            record["methylation_file_id"]
        )

        file_path = record[
            "methylation_local_path"
        ]

        try:
            payload = read_methylation_payload(
                file_path
            )

            observed_probe_ids = pd.Index(
                payload["probe_id"],
                name="probe_id",
            )

            if not observed_probe_ids.equals(
                reference_probe_ids
            ):
                raise ValueError(
                    "Probe identity or order does not "
                    "match the platform reference."
                )

            missing_beta_counts += (
                payload["beta_value"]
                .isna()
                .to_numpy(dtype=np.uint32)
            )

            processed_file_ids.append(
                file_id
            )

        except (Exception, KeyboardInterrupt):
            save_probe_qc_checkpoint(
                platform=platform,
                processed_file_ids=processed_file_ids,
                missing_beta_counts=missing_beta_counts,
            )

            raise RuntimeError(
                "Probe-level QC stopped while processing "
                f"{file_id}: "
                f"{project_relative_path(file_path)}"
            )

        completed_file_count = len(
            processed_file_ids
        )

        checkpoint_due = (
            completed_file_count
            % PROBE_QC_CHECKPOINT_INTERVAL
            == 0
        )

        processing_complete = (
            completed_file_count
            == total_file_count
        )

        if checkpoint_due or processing_complete:
            save_probe_qc_checkpoint(
                platform=platform,
                processed_file_ids=processed_file_ids,
                missing_beta_counts=missing_beta_counts,
            )

            session_elapsed_seconds = (
                pd.Timestamp.now(tz="UTC")
                - session_start_time
            ).total_seconds()

            session_processed_count = (
                completed_file_count
                - processed_file_count
            )

            session_rate = (
                session_processed_count
                / session_elapsed_seconds
                if session_elapsed_seconds > 0
                else np.nan
            )

            remaining_file_count = (
                total_file_count
                - completed_file_count
            )

            remaining_minutes = (
                remaining_file_count
                / session_rate
                / 60
                if session_rate > 0
                else np.nan
            )

            print(
                f"Processed "
                f"{completed_file_count:,}/"
                f"{total_file_count:,} files "
                f"({session_rate:,.2f} files/s; "
                f"ETA {remaining_minutes:,.1f} min)"
            )

    print(
        "Probe-level QC completed for "
        f"{platform}."
    )

    return {
        "platform": platform,
        "processed_file_ids": processed_file_ids,
        "missing_beta_counts": missing_beta_counts,
    }


# -----------------------------------------------------------------------------
# Inspect current checkpoint states without processing payloads
# -----------------------------------------------------------------------------

print("Current probe-level QC checkpoint states:")

for platform in PLATFORM_SLUGS:
    checkpoint = load_probe_qc_checkpoint(
        platform
    )

    processed_count = len(
        checkpoint["processed_file_ids"]
    )

    total_count = len(
        probe_qc_platform_inventories[
            platform
        ]
    )

    print(
        f"{platform}: "
        f"{processed_count:,}/{total_count:,} files"
    )

Current probe-level QC checkpoint states:
Illumina Human Methylation 27: 1,621/1,621 files
Illumina Human Methylation 450: 8,415/8,415 files


In [45]:
# =============================================================================
# Run resumable probe-level methylation QC
# =============================================================================

probe_qc_results = {}

for platform in [
    HM27_PLATFORM,
    HM450_PLATFORM,
]:
    print("=" * 80)

    probe_qc_results[platform] = (
        process_probe_qc_platform(platform)
    )

    print()

print("=" * 80)
print("Probe-level methylation QC completed for all platforms.")

for platform, result in probe_qc_results.items():
    processed_file_count = len(
        result["processed_file_ids"]
    )

    print(
        f"{platform}: "
        f"{processed_file_count:,} files processed"
    )

Platform: Illumina Human Methylation 27
Checkpoint progress: 1,621/1,621
Probe-level QC is already complete.

Platform: Illumina Human Methylation 450
Checkpoint progress: 8,415/8,415
Probe-level QC is already complete.

Probe-level methylation QC completed for all platforms.
Illumina Human Methylation 27: 1,621 files processed
Illumina Human Methylation 450: 8,415 files processed


In [47]:
# =============================================================================
# Construct platform-specific probe-level QC metrics
# =============================================================================

probe_level_qc_frames = []

for platform in [
    HM27_PLATFORM,
    HM450_PLATFORM,
]:
    probe_ids = platform_reference_probe_ids[
        platform
    ]

    missing_beta_counts = np.asarray(
        probe_qc_results[
            platform
        ]["missing_beta_counts"],
        dtype=np.uint32,
    )

    platform_file_count = len(
        probe_qc_platform_inventories[
            platform
        ]
    )

    observed_beta_counts = (
        platform_file_count
        - missing_beta_counts
    )

    platform_probe_qc = pd.DataFrame(
        {
            "methylation_platform": platform,
            "probe_id": probe_ids,
            "platform_file_count": platform_file_count,
            "missing_beta_count": missing_beta_counts,
            "observed_beta_count": observed_beta_counts,
        }
    )

    platform_probe_qc[
        "missing_beta_fraction"
    ] = (
        platform_probe_qc[
            "missing_beta_count"
        ]
        / platform_file_count
    )

    probe_level_qc_frames.append(
        platform_probe_qc
    )

probe_level_qc = pd.concat(
    probe_level_qc_frames,
    ignore_index=True,
)

probe_level_qc = probe_level_qc.merge(
    probe_space_inventory[
        [
            "probe_id",
            "probe_space_status",
        ]
    ],
    on="probe_id",
    how="left",
    validate="many_to_one",
)


# -----------------------------------------------------------------------------
# Validate probe-level metrics
# -----------------------------------------------------------------------------

expected_probe_level_rows = sum(
    len(
        platform_reference_probe_ids[
            platform
        ]
    )
    for platform in [
        HM27_PLATFORM,
        HM450_PLATFORM,
    ]
)

probe_level_qc_checks = {
    "row_count_matches_platform_probe_spaces": (
        len(probe_level_qc)
        == expected_probe_level_rows
    ),
    "platform_probe_pairs_are_unique": (
        not probe_level_qc.duplicated(
            [
                "methylation_platform",
                "probe_id",
            ]
        ).any()
    ),
    "probe_space_membership_is_complete": (
        probe_level_qc[
            "probe_space_status"
        ]
        .notna()
        .all()
    ),
    "counts_reconstruct_platform_file_count": (
        (
            probe_level_qc[
                "missing_beta_count"
            ]
            + probe_level_qc[
                "observed_beta_count"
            ]
        )
        .eq(
            probe_level_qc[
                "platform_file_count"
            ]
        )
        .all()
    ),
    "missing_counts_do_not_exceed_file_count": (
        probe_level_qc[
            "missing_beta_count"
        ]
        .le(
            probe_level_qc[
                "platform_file_count"
            ]
        )
        .all()
    ),
    "missing_fractions_are_valid": (
        probe_level_qc[
            "missing_beta_fraction"
        ]
        .between(
            0,
            1,
            inclusive="both",
        )
        .all()
    ),
}

print("Platform-specific probe-level QC metrics constructed.")
print(
    f"Rows: {len(probe_level_qc):,}"
)

print("\nRows by platform:")
print(
    probe_level_qc[
        "methylation_platform"
    ].value_counts()
)

print("\nProbe-level QC checks:")
for check_name, check_passed in probe_level_qc_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(probe_level_qc_checks.values()):
    raise ValueError(
        "Platform-specific probe-level QC metrics are inconsistent."
    )

Platform-specific probe-level QC metrics constructed.
Rows: 514,005

Rows by platform:
methylation_platform
Illumina Human Methylation 450    486427
Illumina Human Methylation 27      27578
Name: count, dtype: int64

Probe-level QC checks:
row_count_matches_platform_probe_spaces: True
platform_probe_pairs_are_unique: True
probe_space_membership_is_complete: True
counts_reconstruct_platform_file_count: True
missing_counts_do_not_exceed_file_count: True
missing_fractions_are_valid: True


In [49]:
# =============================================================================
# Characterize platform-specific probe missingness
# =============================================================================

PROBE_MISSINGNESS_QUANTILES = [
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    1.00,
]

PROBE_MISSINGNESS_THRESHOLDS = [
    0.01,
    0.05,
    0.10,
    0.20,
    0.50,
    0.90,
    1.00,
]


# -----------------------------------------------------------------------------
# Distribution quantiles
# -----------------------------------------------------------------------------

probe_missingness_quantiles = (
    probe_level_qc
    .groupby(
        "methylation_platform",
        observed=True,
    )["missing_beta_fraction"]
    .quantile(
        PROBE_MISSINGNESS_QUANTILES
    )
    .rename("missing_beta_fraction")
    .reset_index()
    .rename(
        columns={
            "level_1": "quantile",
        }
    )
)


# -----------------------------------------------------------------------------
# Threshold prevalence
# -----------------------------------------------------------------------------

probe_missingness_tail_records = []

for platform, platform_qc in probe_level_qc.groupby(
    "methylation_platform",
    observed=True,
):
    platform_probe_count = len(
        platform_qc
    )

    complete_probe_count = (
        platform_qc[
            "missing_beta_count"
        ]
        .eq(0)
        .sum()
    )

    probe_missingness_tail_records.append(
        {
            "methylation_platform": platform,
            "criterion": "missing_fraction == 0",
            "probe_count": complete_probe_count,
            "probe_fraction": (
                complete_probe_count
                / platform_probe_count
            ),
        }
    )

    for threshold in PROBE_MISSINGNESS_THRESHOLDS:
        affected_probe_count = (
            platform_qc[
                "missing_beta_fraction"
            ]
            .ge(threshold)
            .sum()
        )

        probe_missingness_tail_records.append(
            {
                "methylation_platform": platform,
                "criterion": (
                    f"missing_fraction >= {threshold:.0%}"
                ),
                "probe_count": affected_probe_count,
                "probe_fraction": (
                    affected_probe_count
                    / platform_probe_count
                ),
            }
        )

probe_missingness_tail_summary = pd.DataFrame(
    probe_missingness_tail_records
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("Probe-level missingness quantiles:")
print(
    probe_missingness_quantiles.to_string(
        index=False
    )
)

print("\nProbe-level missingness prevalence:")
print(
    probe_missingness_tail_summary.to_string(
        index=False
    )
)

Probe-level missingness quantiles:
          methylation_platform  quantile  missing_beta_fraction
 Illumina Human Methylation 27      0.00               0.000000
 Illumina Human Methylation 27      0.01               0.000000
 Illumina Human Methylation 27      0.05               0.000000
 Illumina Human Methylation 27      0.25               0.000000
 Illumina Human Methylation 27      0.50               0.000000
 Illumina Human Methylation 27      0.75               0.004318
 Illumina Human Methylation 27      0.95               1.000000
 Illumina Human Methylation 27      0.99               1.000000
 Illumina Human Methylation 27      1.00               1.000000
Illumina Human Methylation 450      0.00               0.000000
Illumina Human Methylation 450      0.01               0.000000
Illumina Human Methylation 450      0.05               0.000000
Illumina Human Methylation 450      0.25               0.000000
Illumina Human Methylation 450      0.50               0.000238
Illum

In [50]:
# =============================================================================
# Characterize universally missing probes
# =============================================================================

probe_level_qc["probe_id_class"] = np.select(
    [
        probe_level_qc["probe_id"].str.startswith(
            "cg",
            na=False,
        ),
        probe_level_qc["probe_id"].str.startswith(
            "rs",
            na=False,
        ),
    ],
    [
        "cg",
        "rs",
    ],
    default="other",
)

probe_level_qc["universally_missing"] = (
    probe_level_qc["missing_beta_fraction"]
    .eq(1.0)
)


# -----------------------------------------------------------------------------
# Universal missingness by platform and probe class
# -----------------------------------------------------------------------------

universal_missingness_by_class = (
    probe_level_qc
    .groupby(
        [
            "methylation_platform",
            "probe_id_class",
        ],
        observed=True,
    )
    .agg(
        probe_count=(
            "probe_id",
            "size",
        ),
        universally_missing_count=(
            "universally_missing",
            "sum",
        ),
    )
    .reset_index()
)

universal_missingness_by_class[
    "universally_missing_fraction"
] = (
    universal_missingness_by_class[
        "universally_missing_count"
    ]
    / universal_missingness_by_class[
        "probe_count"
    ]
)


# -----------------------------------------------------------------------------
# Universal missingness by platform and probe-space membership
# -----------------------------------------------------------------------------

universal_missingness_by_space = (
    probe_level_qc
    .groupby(
        [
            "methylation_platform",
            "probe_space_status",
        ],
        observed=True,
    )
    .agg(
        probe_count=(
            "probe_id",
            "size",
        ),
        universally_missing_count=(
            "universally_missing",
            "sum",
        ),
    )
    .reset_index()
)

universal_missingness_by_space[
    "universally_missing_fraction"
] = (
    universal_missingness_by_space[
        "universally_missing_count"
    ]
    / universal_missingness_by_space[
        "probe_count"
    ]
)


# -----------------------------------------------------------------------------
# Shared-probe cross-platform missingness status
# -----------------------------------------------------------------------------

shared_probe_missingness = (
    probe_level_qc.loc[
        probe_level_qc[
            "probe_space_status"
        ].eq("shared_hm27_hm450"),
        [
            "methylation_platform",
            "probe_id",
            "universally_missing",
        ],
    ]
    .pivot(
        index="probe_id",
        columns="methylation_platform",
        values="universally_missing",
    )
    .reset_index()
)

shared_probe_missingness[
    "shared_universal_missingness_status"
] = np.select(
    [
        (
            shared_probe_missingness[
                HM27_PLATFORM
            ]
            & shared_probe_missingness[
                HM450_PLATFORM
            ]
        ),
        (
            shared_probe_missingness[
                HM27_PLATFORM
            ]
            & ~shared_probe_missingness[
                HM450_PLATFORM
            ]
        ),
        (
            ~shared_probe_missingness[
                HM27_PLATFORM
            ]
            & shared_probe_missingness[
                HM450_PLATFORM
            ]
        ),
    ],
    [
        "universally_missing_both",
        "universally_missing_hm27_only",
        "universally_missing_hm450_only",
    ],
    default="not_universally_missing",
)

shared_universal_missingness_summary = (
    shared_probe_missingness[
        "shared_universal_missingness_status"
    ]
    .value_counts()
    .rename_axis(
        "shared_universal_missingness_status"
    )
    .reset_index(
        name="probe_count"
    )
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("Universal missingness by probe class:")
print(
    universal_missingness_by_class.to_string(
        index=False
    )
)

print("\nUniversal missingness by probe-space membership:")
print(
    universal_missingness_by_space.to_string(
        index=False
    )
)

print("\nShared-probe cross-platform universal missingness:")
print(
    shared_universal_missingness_summary.to_string(
        index=False
    )
)

Universal missingness by probe class:
          methylation_platform probe_id_class  probe_count  universally_missing_count  universally_missing_fraction
 Illumina Human Methylation 27             cg        27578                       2263                      0.082058
Illumina Human Methylation 450             cg       482421                      63563                      0.131758
Illumina Human Methylation 450          other         3941                        766                      0.194367
Illumina Human Methylation 450             rs           65                          0                           0.0

Universal missingness by probe-space membership:
          methylation_platform probe_space_status  probe_count  universally_missing_count  universally_missing_fraction
 Illumina Human Methylation 27          hm27_only         1600                        645                      0.403125
 Illumina Human Methylation 27  shared_hm27_hm450        25978                       1618   

In [51]:
# =============================================================================
# Characterize non-universally-missing probe distributions
# =============================================================================

NON_UNIVERSAL_MISSINGNESS_QUANTILES = [
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00,
]

NON_UNIVERSAL_MISSINGNESS_THRESHOLDS = [
    0.01,
    0.05,
    0.10,
    0.20,
    0.50,
    0.90,
]

non_universal_probe_qc = (
    probe_level_qc.loc[
        ~probe_level_qc["universally_missing"]
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Platform-specific distributions
# -----------------------------------------------------------------------------

non_universal_missingness_quantiles = (
    non_universal_probe_qc
    .groupby(
        "methylation_platform",
        observed=True,
    )["missing_beta_fraction"]
    .quantile(
        NON_UNIVERSAL_MISSINGNESS_QUANTILES
    )
    .rename("missing_beta_fraction")
    .reset_index()
    .rename(
        columns={
            "level_1": "quantile",
        }
    )
)

non_universal_tail_records = []

for platform, platform_qc in non_universal_probe_qc.groupby(
    "methylation_platform",
    observed=True,
):
    platform_probe_count = len(
        platform_qc
    )

    for threshold in NON_UNIVERSAL_MISSINGNESS_THRESHOLDS:
        affected_probe_count = (
            platform_qc[
                "missing_beta_fraction"
            ]
            .ge(threshold)
            .sum()
        )

        non_universal_tail_records.append(
            {
                "methylation_platform": platform,
                "criterion": (
                    f"missing_fraction >= {threshold:.0%}"
                ),
                "probe_count": affected_probe_count,
                "probe_fraction": (
                    affected_probe_count
                    / platform_probe_count
                ),
            }
        )

non_universal_tail_summary = pd.DataFrame(
    non_universal_tail_records
)


# -----------------------------------------------------------------------------
# Shared-probe cross-platform concordance
# -----------------------------------------------------------------------------

shared_probe_missingness_wide = (
    probe_level_qc.loc[
        probe_level_qc[
            "probe_space_status"
        ].eq("shared_hm27_hm450"),
        [
            "probe_id",
            "methylation_platform",
            "missing_beta_fraction",
        ],
    ]
    .pivot(
        index="probe_id",
        columns="methylation_platform",
        values="missing_beta_fraction",
    )
)

shared_probe_missingness_wide = (
    shared_probe_missingness_wide.loc[
        shared_probe_missingness_wide[
            HM27_PLATFORM
        ].lt(1.0)
        & shared_probe_missingness_wide[
            HM450_PLATFORM
        ].lt(1.0)
    ]
)

shared_missingness_pearson = (
    shared_probe_missingness_wide[
        HM27_PLATFORM
    ]
    .corr(
        shared_probe_missingness_wide[
            HM450_PLATFORM
        ],
        method="pearson",
    )
)

shared_missingness_spearman = (
    shared_probe_missingness_wide[
        HM27_PLATFORM
    ]
    .corr(
        shared_probe_missingness_wide[
            HM450_PLATFORM
        ],
        method="spearman",
    )
)

shared_missingness_absolute_difference = (
    shared_probe_missingness_wide[
        HM27_PLATFORM
    ]
    - shared_probe_missingness_wide[
        HM450_PLATFORM
    ]
).abs()

shared_difference_quantiles = (
    shared_missingness_absolute_difference
    .quantile(
        [
            0.50,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
    .rename("absolute_missingness_difference")
    .reset_index()
    .rename(
        columns={
            "index": "quantile",
        }
    )
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("Non-universally-missing probe quantiles:")
print(
    non_universal_missingness_quantiles.to_string(
        index=False
    )
)

print("\nNon-universally-missing probe prevalence:")
print(
    non_universal_tail_summary.to_string(
        index=False
    )
)

print("\nShared-probe missingness concordance:")
print(
    f"Shared probes evaluated: "
    f"{len(shared_probe_missingness_wide):,}"
)
print(
    f"Pearson correlation: "
    f"{shared_missingness_pearson:.4f}"
)
print(
    f"Spearman correlation: "
    f"{shared_missingness_spearman:.4f}"
)

print("\nAbsolute cross-platform missingness differences:")
print(
    shared_difference_quantiles.to_string(
        index=False
    )
)

Non-universally-missing probe quantiles:
          methylation_platform  quantile  missing_beta_fraction
 Illumina Human Methylation 27      0.00               0.000000
 Illumina Human Methylation 27      0.01               0.000000
 Illumina Human Methylation 27      0.05               0.000000
 Illumina Human Methylation 27      0.25               0.000000
 Illumina Human Methylation 27      0.50               0.000000
 Illumina Human Methylation 27      0.75               0.001234
 Illumina Human Methylation 27      0.90               0.031462
 Illumina Human Methylation 27      0.95               0.141271
 Illumina Human Methylation 27      0.99               0.683529
 Illumina Human Methylation 27      1.00               0.999383
Illumina Human Methylation 450      0.00               0.000000
Illumina Human Methylation 450      0.01               0.000000
Illumina Human Methylation 450      0.05               0.000000
Illumina Human Methylation 450      0.25               0.000000

In [52]:
# =============================================================================
# Compare candidate probe-missingness policies
# =============================================================================

CANDIDATE_PROBE_MISSINGNESS_THRESHOLDS = [
    0.10,
    0.20,
    0.50,
]

probe_policy_comparison_records = []

for threshold in CANDIDATE_PROBE_MISSINGNESS_THRESHOLDS:
    for platform, platform_qc in probe_level_qc.groupby(
        "methylation_platform",
        observed=True,
    ):
        eligible_probe_mask = (
            platform_qc["missing_beta_fraction"]
            < threshold
        )

        eligible_probe_count = int(
            eligible_probe_mask.sum()
        )

        total_probe_count = len(
            platform_qc
        )

        probe_policy_comparison_records.append(
            {
                "policy_scope": platform,
                "maximum_missing_fraction": threshold,
                "total_probe_count": total_probe_count,
                "eligible_probe_count": eligible_probe_count,
                "excluded_probe_count": (
                    total_probe_count
                    - eligible_probe_count
                ),
                "eligible_probe_fraction": (
                    eligible_probe_count
                    / total_probe_count
                ),
            }
        )

    shared_policy_qc = (
        probe_level_qc.loc[
            probe_level_qc[
                "probe_space_status"
            ].eq("shared_hm27_hm450"),
            [
                "probe_id",
                "methylation_platform",
                "missing_beta_fraction",
            ],
        ]
        .pivot(
            index="probe_id",
            columns="methylation_platform",
            values="missing_beta_fraction",
        )
    )

    shared_eligible_mask = (
        shared_policy_qc[HM27_PLATFORM].lt(threshold)
        & shared_policy_qc[HM450_PLATFORM].lt(threshold)
    )

    shared_eligible_count = int(
        shared_eligible_mask.sum()
    )

    shared_total_count = len(
        shared_policy_qc
    )

    probe_policy_comparison_records.append(
        {
            "policy_scope": "shared_hm27_hm450_both_platforms",
            "maximum_missing_fraction": threshold,
            "total_probe_count": shared_total_count,
            "eligible_probe_count": shared_eligible_count,
            "excluded_probe_count": (
                shared_total_count
                - shared_eligible_count
            ),
            "eligible_probe_fraction": (
                shared_eligible_count
                / shared_total_count
            ),
        }
    )

probe_policy_comparison = pd.DataFrame(
    probe_policy_comparison_records
)

print("Candidate probe-missingness policy comparison:")
print(
    probe_policy_comparison.to_string(
        index=False,
        formatters={
            "maximum_missing_fraction": "{:.0%}".format,
            "eligible_probe_fraction": "{:.2%}".format,
        },
    )
)

Candidate probe-missingness policy comparison:
                    policy_scope maximum_missing_fraction  total_probe_count  eligible_probe_count  excluded_probe_count eligible_probe_fraction
   Illumina Human Methylation 27                      10%              27578                 23775                  3803                  86.21%
  Illumina Human Methylation 450                      10%             486427                397148                 89279                  81.65%
shared_hm27_hm450_both_platforms                      10%              25978                 22778                  3200                  87.68%
   Illumina Human Methylation 27                      20%              27578                 24303                  3275                  88.12%
  Illumina Human Methylation 450                      20%             486427                407696                 78731                  83.81%
shared_hm27_hm450_both_platforms                      20%              25978       

In [53]:
# =============================================================================
# Apply the primary probe-level missingness policy
# =============================================================================

PRIMARY_PROBE_MISSINGNESS_THRESHOLD = 0.20

probe_level_qc[
    "probe_qc_missingness_threshold"
] = PRIMARY_PROBE_MISSINGNESS_THRESHOLD

probe_level_qc[
    "probe_qc_eligible_within_platform"
] = (
    probe_level_qc["missing_beta_fraction"]
    .lt(PRIMARY_PROBE_MISSINGNESS_THRESHOLD)
)

probe_level_qc[
    "probe_qc_eligibility_status"
] = np.where(
    probe_level_qc[
        "probe_qc_eligible_within_platform"
    ],
    "Eligible — probe missingness below 20%",
    "Ineligible — probe missingness at or above 20%",
)

probe_level_qc[
    "probe_qc_ineligibility_reason"
] = np.select(
    [
        probe_level_qc["universally_missing"],
        (
            ~probe_level_qc["universally_missing"]
            & ~probe_level_qc[
                "probe_qc_eligible_within_platform"
            ]
        ),
    ],
    [
        (
            "No observed beta values among eligible "
            "files from this platform"
        ),
        (
            "Missing beta fraction is at or above "
            "the primary 20% platform-specific threshold"
        ),
    ],
    default="",
)


# -----------------------------------------------------------------------------
# Construct the shared probe space passing on both platforms
# -----------------------------------------------------------------------------

shared_probe_rows = probe_level_qc.loc[
    probe_level_qc[
        "probe_space_status"
    ].eq("shared_hm27_hm450")
]

shared_missingness_wide = shared_probe_rows.pivot(
    index="probe_id",
    columns="methylation_platform",
    values="missing_beta_fraction",
)

shared_eligibility_wide = shared_probe_rows.pivot(
    index="probe_id",
    columns="methylation_platform",
    values="probe_qc_eligible_within_platform",
)

shared_probe_qc_inventory = pd.DataFrame(
    {
        "probe_id": shared_missingness_wide.index,
        "hm27_missing_beta_fraction": (
            shared_missingness_wide[
                HM27_PLATFORM
            ].to_numpy()
        ),
        "hm450_missing_beta_fraction": (
            shared_missingness_wide[
                HM450_PLATFORM
            ].to_numpy()
        ),
        "hm27_probe_qc_eligible": (
            shared_eligibility_wide[
                HM27_PLATFORM
            ].to_numpy()
        ),
        "hm450_probe_qc_eligible": (
            shared_eligibility_wide[
                HM450_PLATFORM
            ].to_numpy()
        ),
    }
)

shared_probe_qc_inventory[
    "shared_probe_qc_eligible"
] = (
    shared_probe_qc_inventory[
        "hm27_probe_qc_eligible"
    ]
    & shared_probe_qc_inventory[
        "hm450_probe_qc_eligible"
    ]
)


# -----------------------------------------------------------------------------
# Validate and summarize the policy
# -----------------------------------------------------------------------------

probe_policy_checks = {
    "all_platform_probe_statuses_are_complete": (
        probe_level_qc[
            "probe_qc_eligibility_status"
        ]
        .notna()
        .all()
    ),
    "eligible_probes_have_no_ineligibility_reason": (
        probe_level_qc.loc[
            probe_level_qc[
                "probe_qc_eligible_within_platform"
            ],
            "probe_qc_ineligibility_reason",
        ]
        .eq("")
        .all()
    ),
    "ineligible_probes_have_a_reason": (
        probe_level_qc.loc[
            ~probe_level_qc[
                "probe_qc_eligible_within_platform"
            ],
            "probe_qc_ineligibility_reason",
        ]
        .ne("")
        .all()
    ),
    "shared_probe_inventory_is_unique": (
        shared_probe_qc_inventory[
            "probe_id"
        ]
        .is_unique
    ),
    "shared_probe_count_matches_reference": (
        len(shared_probe_qc_inventory)
        == len(shared_probe_ids)
    ),
}

platform_policy_summary = (
    probe_level_qc
    .groupby(
        "methylation_platform",
        observed=True,
    )
    .agg(
        total_probe_count=("probe_id", "size"),
        eligible_probe_count=(
            "probe_qc_eligible_within_platform",
            "sum",
        ),
    )
    .reset_index()
)

platform_policy_summary[
    "excluded_probe_count"
] = (
    platform_policy_summary["total_probe_count"]
    - platform_policy_summary["eligible_probe_count"]
)

platform_policy_summary[
    "eligible_probe_fraction"
] = (
    platform_policy_summary["eligible_probe_count"]
    / platform_policy_summary["total_probe_count"]
)

shared_eligible_probe_count = int(
    shared_probe_qc_inventory[
        "shared_probe_qc_eligible"
    ].sum()
)

print("Primary probe-level missingness policy applied.")
print(
    "Eligibility criterion: "
    "platform-specific missingness < 20%"
)

print("\nPlatform-specific results:")
print(
    platform_policy_summary.to_string(
        index=False,
        formatters={
            "eligible_probe_fraction": "{:.2%}".format,
        },
    )
)

print("\nShared cross-platform space:")
print(
    f"Total shared probes: "
    f"{len(shared_probe_qc_inventory):,}"
)
print(
    f"Eligible on both platforms: "
    f"{shared_eligible_probe_count:,}"
)

print("\nProbe-policy checks:")
for check_name, check_passed in probe_policy_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(probe_policy_checks.values()):
    raise ValueError(
        "Probe-level missingness policy is inconsistent."
    )

Primary probe-level missingness policy applied.
Eligibility criterion: platform-specific missingness < 20%

Platform-specific results:
          methylation_platform  total_probe_count  eligible_probe_count  excluded_probe_count eligible_probe_fraction
 Illumina Human Methylation 27              27578                 24303                  3275                0.881246
Illumina Human Methylation 450             486427                407696                 78731                0.838144

Shared cross-platform space:
Total shared probes: 25,978
Eligible on both platforms: 23,356

Probe-policy checks:
all_platform_probe_statuses_are_complete: True
eligible_probes_have_no_ineligibility_reason: True
ineligible_probes_have_a_reason: True
shared_probe_inventory_is_unique: True
shared_probe_count_matches_reference: True


In [55]:
# =============================================================================
# Save and validate probe-level methylation QC artifacts
# =============================================================================

PROBE_LEVEL_QC_METRICS_PATH = (
    METHYLATION_QC_OUTPUT_DIR
    / (
        "tcga_primary_tumor_methylation_hm27_hm450_"
        "probe_qc_metrics.csv"
    )
)

SHARED_PROBE_QC_INVENTORY_PATH = (
    METHYLATION_METADATA_OUTPUT_DIR
    / (
        "tcga_primary_tumor_methylation_hm27_hm450_"
        "shared_probe_qc_inventory.csv"
    )
)


# -----------------------------------------------------------------------------
# Prepare deterministic output order
# -----------------------------------------------------------------------------

probe_level_qc_output = probe_level_qc.copy()

probe_level_qc_output[
    "platform_probe_order"
] = (
    probe_level_qc_output
    .groupby(
        "methylation_platform",
        sort=False,
        observed=True,
    )
    .cumcount()
)

probe_level_qc_output = probe_level_qc_output[
    [
        "methylation_platform",
        "platform_probe_order",
        "probe_id",
        "probe_id_class",
        "probe_space_status",
        "platform_file_count",
        "observed_beta_count",
        "missing_beta_count",
        "missing_beta_fraction",
        "universally_missing",
        "probe_qc_missingness_threshold",
        "probe_qc_eligible_within_platform",
        "probe_qc_eligibility_status",
        "probe_qc_ineligibility_reason",
    ]
]

shared_probe_order = pd.Series(
    np.arange(
        len(shared_probe_ids),
        dtype=np.int64,
    ),
    index=shared_probe_ids,
)

shared_probe_qc_output = (
    shared_probe_qc_inventory
    .copy()
)

shared_probe_qc_output[
    "shared_probe_order"
] = (
    shared_probe_qc_output[
        "probe_id"
    ]
    .map(shared_probe_order)
)

shared_probe_qc_output[
    "probe_qc_missingness_threshold"
] = PRIMARY_PROBE_MISSINGNESS_THRESHOLD

shared_probe_qc_output = (
    shared_probe_qc_output
    .sort_values(
        "shared_probe_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

shared_probe_qc_output = shared_probe_qc_output[
    [
        "shared_probe_order",
        "probe_id",
        "hm27_missing_beta_fraction",
        "hm450_missing_beta_fraction",
        "probe_qc_missingness_threshold",
        "hm27_probe_qc_eligible",
        "hm450_probe_qc_eligible",
        "shared_probe_qc_eligible",
    ]
]


# -----------------------------------------------------------------------------
# Pre-write validation
# -----------------------------------------------------------------------------

pre_write_checks = {
    "probe_metrics_are_not_empty": (
        not probe_level_qc_output.empty
    ),
    "probe_metrics_have_unique_platform_probe_pairs": (
        not probe_level_qc_output.duplicated(
            [
                "methylation_platform",
                "probe_id",
            ]
        ).any()
    ),
    "platform_probe_order_is_unique": (
        not probe_level_qc_output.duplicated(
            [
                "methylation_platform",
                "platform_probe_order",
            ]
        ).any()
    ),
    "shared_inventory_is_not_empty": (
        not shared_probe_qc_output.empty
    ),
    "shared_probe_ids_are_unique": (
        shared_probe_qc_output[
            "probe_id"
        ].is_unique
    ),
    "shared_probe_order_is_complete": (
        shared_probe_qc_output[
            "shared_probe_order"
        ]
        .notna()
        .all()
    ),
    "shared_probe_order_is_unique": (
        shared_probe_qc_output[
            "shared_probe_order"
        ].is_unique
    ),
    "shared_eligible_count_matches_policy": (
        shared_probe_qc_output[
            "shared_probe_qc_eligible"
        ].sum()
        == shared_eligible_probe_count
    ),
}

print("Probe-level QC pre-write checks:")
for check_name, check_passed in pre_write_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(pre_write_checks.values()):
    raise ValueError(
        "Probe-level QC artifacts failed pre-write validation."
    )


# -----------------------------------------------------------------------------
# Atomic writes
# -----------------------------------------------------------------------------

METHYLATION_QC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METHYLATION_METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

probe_metrics_temporary_path = (
    PROBE_LEVEL_QC_METRICS_PATH.with_suffix(
        ".tmp.csv"
    )
)

shared_inventory_temporary_path = (
    SHARED_PROBE_QC_INVENTORY_PATH.with_suffix(
        ".tmp.csv"
    )
)

probe_level_qc_output.to_csv(
    probe_metrics_temporary_path,
    index=False,
)

shared_probe_qc_output.to_csv(
    shared_inventory_temporary_path,
    index=False,
)

probe_metrics_temporary_path.replace(
    PROBE_LEVEL_QC_METRICS_PATH
)

shared_inventory_temporary_path.replace(
    SHARED_PROBE_QC_INVENTORY_PATH
)


# -----------------------------------------------------------------------------
# Round-trip validation
# -----------------------------------------------------------------------------

written_probe_level_qc = pd.read_csv(
    PROBE_LEVEL_QC_METRICS_PATH
)

written_shared_probe_qc = pd.read_csv(
    SHARED_PROBE_QC_INVENTORY_PATH
)

# -----------------------------------------------------------------------------
# Round-trip validation
# -----------------------------------------------------------------------------

written_probe_level_qc = pd.read_csv(
    PROBE_LEVEL_QC_METRICS_PATH
)

written_shared_probe_qc = pd.read_csv(
    SHARED_PROBE_QC_INVENTORY_PATH
)


# -----------------------------------------------------------------------------
# Compare values independently of CSV dtype coercion
# -----------------------------------------------------------------------------

probe_platform_matches = np.array_equal(
    written_probe_level_qc[
        "methylation_platform"
    ].astype(str).to_numpy(),
    probe_level_qc_output[
        "methylation_platform"
    ].astype(str).to_numpy(),
)

platform_probe_order_matches = np.array_equal(
    written_probe_level_qc[
        "platform_probe_order"
    ].to_numpy(dtype=np.int64),
    probe_level_qc_output[
        "platform_probe_order"
    ].to_numpy(dtype=np.int64),
)

probe_id_matches = np.array_equal(
    written_probe_level_qc[
        "probe_id"
    ].astype(str).to_numpy(),
    probe_level_qc_output[
        "probe_id"
    ].astype(str).to_numpy(),
)

probe_missing_counts_match = np.array_equal(
    written_probe_level_qc[
        "missing_beta_count"
    ].to_numpy(dtype=np.int64),
    probe_level_qc_output[
        "missing_beta_count"
    ].to_numpy(dtype=np.int64),
)

probe_eligibility_matches = np.array_equal(
    written_probe_level_qc[
        "probe_qc_eligible_within_platform"
    ].astype(str).to_numpy(),
    probe_level_qc_output[
        "probe_qc_eligible_within_platform"
    ].astype(str).to_numpy(),
)

shared_probe_order_matches = np.array_equal(
    written_shared_probe_qc[
        "shared_probe_order"
    ].to_numpy(dtype=np.int64),
    shared_probe_qc_output[
        "shared_probe_order"
    ].to_numpy(dtype=np.int64),
)

shared_probe_id_matches = np.array_equal(
    written_shared_probe_qc[
        "probe_id"
    ].astype(str).to_numpy(),
    shared_probe_qc_output[
        "probe_id"
    ].astype(str).to_numpy(),
)

shared_eligibility_matches = np.array_equal(
    written_shared_probe_qc[
        "shared_probe_qc_eligible"
    ].astype(str).to_numpy(),
    shared_probe_qc_output[
        "shared_probe_qc_eligible"
    ].astype(str).to_numpy(),
)


# -----------------------------------------------------------------------------
# Written-artifact checks
# -----------------------------------------------------------------------------

written_checks = {
    "probe_metrics_file_exists": (
        PROBE_LEVEL_QC_METRICS_PATH.is_file()
    ),
    "shared_inventory_file_exists": (
        SHARED_PROBE_QC_INVENTORY_PATH.is_file()
    ),
    "probe_metrics_shape_matches_memory": (
        written_probe_level_qc.shape
        == probe_level_qc_output.shape
    ),
    "probe_metrics_columns_match_memory": (
        written_probe_level_qc.columns.tolist()
        == probe_level_qc_output.columns.tolist()
    ),
    "probe_platform_matches_memory": (
        probe_platform_matches
    ),
    "platform_probe_order_matches_memory": (
        platform_probe_order_matches
    ),
    "probe_ids_match_memory": (
        probe_id_matches
    ),
    "probe_missing_counts_match_memory": (
        probe_missing_counts_match
    ),
    "probe_missing_fractions_match_memory": (
        np.allclose(
            written_probe_level_qc[
                "missing_beta_fraction"
            ].to_numpy(dtype=float),
            probe_level_qc_output[
                "missing_beta_fraction"
            ].to_numpy(dtype=float),
            equal_nan=True,
        )
    ),
    "probe_eligibility_matches_memory": (
        probe_eligibility_matches
    ),
    "shared_inventory_shape_matches_memory": (
        written_shared_probe_qc.shape
        == shared_probe_qc_output.shape
    ),
    "shared_probe_order_matches_memory": (
        shared_probe_order_matches
    ),
    "shared_probe_ids_match_memory": (
        shared_probe_id_matches
    ),
    "shared_eligibility_matches_memory": (
        shared_eligibility_matches
    ),
}

print("Written-artifact checks:")
for check_name, check_passed in written_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(written_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in written_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Written probe-level QC artifacts failed validation: "
        + ", ".join(failed_checks)
    )


# -----------------------------------------------------------------------------
# Report
# -----------------------------------------------------------------------------

probe_metrics_size_mib = (
    PROBE_LEVEL_QC_METRICS_PATH.stat().st_size
    / 1024**2
)

shared_inventory_size_mib = (
    SHARED_PROBE_QC_INVENTORY_PATH.stat().st_size
    / 1024**2
)

print("\nProbe-level methylation QC artifacts validated.")

print("\nPlatform-specific probe metrics:")
print(
    "Path: "
    f"{project_relative_path(PROBE_LEVEL_QC_METRICS_PATH)}"
)
print(
    f"Shape: {probe_level_qc_output.shape}"
)
print(
    f"File size: {probe_metrics_size_mib:.3f} MiB"
)

print("\nShared cross-platform probe inventory:")
print(
    "Path: "
    f"{project_relative_path(SHARED_PROBE_QC_INVENTORY_PATH)}"
)
print(
    f"Shape: {shared_probe_qc_output.shape}"
)
print(
    "Eligible on both platforms: "
    f"{shared_eligible_probe_count:,}"
)
print(
    f"File size: {shared_inventory_size_mib:.3f} MiB"
)

Probe-level QC pre-write checks:
probe_metrics_are_not_empty: True
probe_metrics_have_unique_platform_probe_pairs: True
platform_probe_order_is_unique: True
shared_inventory_is_not_empty: True
shared_probe_ids_are_unique: True
shared_probe_order_is_complete: True
shared_probe_order_is_unique: True
shared_eligible_count_matches_policy: True
Written-artifact checks:
probe_metrics_file_exists: True
shared_inventory_file_exists: True
probe_metrics_shape_matches_memory: True
probe_metrics_columns_match_memory: True
probe_platform_matches_memory: True
platform_probe_order_matches_memory: True
probe_ids_match_memory: True
probe_missing_counts_match_memory: True
probe_missing_fractions_match_memory: True
probe_eligibility_matches_memory: True
shared_inventory_shape_matches_memory: True
shared_probe_order_matches_memory: True
shared_probe_ids_match_memory: True
shared_eligibility_matches_memory: True

Probe-level methylation QC artifacts validated.

Platform-specific probe metrics:
Path: data/i

## Methylation QC summary

Analytical quality control was completed for the TCGA primary-tumor SeSAMe methylation files represented in the RNA-seq-QC-eligible candidate-pair inventory.

### File-level QC

- 10,038 unique methylation files were evaluated:
  - 1,621 HM27 files;
  - 8,417 HM450 files.
- All candidate payloads matched the frozen methylation file index and the expected platform-specific probe identity and order.
- Two HM450 files showed comparator-supported technical underperformance, characterized by severe platform-aware missingness together with a same-sample, same-platform aliquot displaying materially better coverage and highly concordant shared beta values.
- After methylation file-level QC:
  - 10,036 methylation files remained eligible;
  - 2 files were excluded;
  - 107 eligible files retained non-exclusionary review flags.
- Propagation to the candidate-pair inventory resulted in:
  - 10,154 candidate pairs eligible after RNA-seq and methylation QC;
  - 4 pairs ineligible because of RNA-seq QC;
  - 4 pairs ineligible because of methylation QC.
- No case or TCGA project was lost through methylation QC.

### Probe-space characterization

The platform reference spaces contained:

- 27,578 HM27 probes;
- 486,427 HM450 probes;
- 25,978 probes shared between HM27 and HM450.

The shared space represents 94.20% of the HM27 probe set but only 5.34% of the HM450 probe set. Platform-specific analyses should therefore preserve the larger HM450 feature space, whereas any direct HM27–HM450 integration must use an explicitly aligned shared-probe space.

### Probe-level QC

Probe missingness was calculated across all methylation-QC-eligible files separately for each platform.

A primary platform-specific eligibility criterion of missing beta fraction below 20% was applied. This threshold was selected as a transparent operational compromise rather than as a biologically intrinsic cutoff. A stricter 10% threshold should be retained as a sensitivity analysis where relevant.

The primary policy retained:

- 24,303 of 27,578 HM27 probes;
- 407,696 of 486,427 HM450 probes;
- 23,356 of 25,978 shared probes on both platforms.

Universally missing probes were classified as non-informative within this frozen cohort and processing context. Their exclusion should not be interpreted as evidence that the probes are intrinsically invalid in other datasets or preprocessing workflows.

### Versioned outputs

- `data/interim/qc/tcga_primary_tumor_methylation_candidate_file_qc_metrics.csv`
- `data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_qc_annotated_candidate_pair_inventory.csv`
- `data/interim/qc/tcga_primary_tumor_methylation_hm27_hm450_probe_qc_metrics.csv`
- `data/interim/metadata/tcga_primary_tumor_methylation_hm27_hm450_shared_probe_qc_inventory.csv`

The checkpoint files under `data/interim/qc/tcga_primary_tumor_methylation_probe_qc_checkpoints/` are operational recovery artifacts and are not downstream analytical outputs.

### Scope boundary

This notebook does not:

- select a final unique multi-omic sample per case;
- construct the methylation beta-value matrix;
- impute missing beta values;
- perform batch correction;
- pool HM27 and HM450 naïvely;
- annotate probes to genes or regulatory regions;
- infer methylation-expression associations.

Final sample selection, matrix construction, platform-aware feature handling, and downstream integration remain separate analytical stages.